# MMRL / BayesMMRL 动态 R 权重融合验证（repo-forward + 真实日志配置恢复 + MC epistemic percentile/rank 可靠性调节版 v9：full-MC sample outputs + rank reliability）

本文件在 v5 基础上进一步修复 `mmrl_dynamic_r_fusion_repo_forward_corrected.ipynb` 的关键问题：

> **不再只从 `case_root` 路径推断配置。**  
> 每个 case 必须从真实训练日志 `run.log` / `log.txt` / `log.txt-*` 的 `** Arguments **` 区块恢复 `dataset_config_file`、`method_config_file`、`protocol_config_file`、`runtime_config_file`、`exp_config`、`exec_mode` 和完整 `opts`。

这很重要，因为 MMRL / BayesMMRL 的 checkpoint 是 lightweight checkpoint，很多行为由当前重建的 cfg 决定。只靠路径恢复 `shots/backbone/subsample` 会漏掉真实训练时的 `BAYES_MMRL.*`、`MMRL.*`、`EVAL_AGGREGATION`、`N_MC_TEST`、`ALPHA` 等覆盖项，导致 z_c / z_r / fusion 的准确率异常。

核心提取路径仍然严格使用 repo 标准评估接口：

```python
outputs = trainer.method.forward_eval(batch, eval_ctx)
z_c = outputs.logits
z_r = outputs.aux_logits["rep"]
z_aux_fusion = outputs.aux_logits["fusion"]
z_repo_eval = trainer.method.select_eval_logits(outputs, eval_ctx)
```

输出文件：

```text
output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored/
  dynamic_r_fusion_repo_forward_log_restored_v9_epistemic_rank_reliability_gate_report.xlsx
```


> v4 改进：加入 `gamma shrink`、校准友好 beta 选择、最终 temperature scaling，并在 Accuracy sheet 中输出置信度/entropy/logit norm 诊断。

## v5/v6 新增：BayesMMRL R 分支 MC 认知不确定性

BayesMMRL 的 repo-forward 收集阶段额外读取 R 分支 MC logits stack，计算 MC predictive mutual information：

```python
MI_R = H(mean_s p_s) - mean_s H(p_s)
```

其中 `p_s = softmax(z_r^{(s)})`。v6 支持 `USE_MEAN_MAIN_MC_REP=False` / full-MC eval 路径，会额外调用 `model.forward_train_samples(image, n_mc)` 收集每个 MC sample 的 `logits_rep`。

## v9 关键改动：用 percentile/rank 替代 raw scaled MI

v8 使用 `tau_e - epistemic_r_scaled`，但由于 MC MI 分布极度长尾，`tau_e=median` 往往很小，导致低 epistemic 的奖励很弱、高 epistemic 的惩罚很强。v9 改为基于验证集分布的 percentile/rank：

```python
epi_rank = percentile_rank_val(epistemic_r_mi)  # 0=最低不确定性, 1=最高不确定性
reliability = 0.5 - epi_rank
score = base + bias + beta_u*u_gap + beta_m*margin_gain + beta_d*JS + lambda_e * reliability
```

因此：

- `epi_rank < 0.5`：R 分支 MC 认知不确定性处在低分位，增加 R 权重；
- `epi_rank > 0.5`：R 分支 MC 认知不确定性处在高分位，降低 R 权重；
- 奖励和惩罚幅度天然对称，避免 raw MI 长尾导致的量纲问题。

v9 还提供两个稳健对照：

- `epistemic_margin_rank_reliability`：只有低 MI 且 R margin 更强时奖励 R；只有高 MI 且 R margin 更弱时惩罚 R；
- `epistemic_high_rank_veto`：只对最高不确定性分位样本做 veto，不奖励低不确定性样本。



In [11]:
from pathlib import Path
import numpy as np

# ====== 改这里 ======
REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATA_ROOT = REPO_ROOT / "DATASETS"

# 每个 case_root 必须是一次真实实验的 seed 目录。
# notebook 会从该目录下的 run.log / log.txt / log.txt-* 恢复真实 Arguments/opts。
da="ucf101" #caltech101 ucf101
CASES = [
    {
        "name": "MMRL_"+da+"_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/MMRL/FS/fewshot_train" /da/ "shots_16/ViT-B-16/default/seed1",
        # 可选：显式指定真实日志文件；不填则自动找 run.log / log.txt / log.txt-*。
        # "log_file": REPO_ROOT / ".../seed1/run.log",
        # 可选：指定加载 epoch；默认加载 model-best 或最后一个 model.pth。
        # "load_epoch": None,
    },
    {
        "name": "BayesMMRL_"+da+"_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/BayesMMRL/FS/fewshot_train" /da/ "shots_16/ViT-B-16/default/seed1",
        # "log_file": REPO_ROOT / ".../seed1/run.log",
    },
]

TEST_SPLIT = "test"
BETA_SELECT_SPLIT = "val"

# 调试时可设成小整数；正式实验设 None。
MAX_BATCHES = None

# β 网格。动态融合部分仅用于分析；分支提取和准确率不依赖 β。
BETA_GRID_VALUES = [-4.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 4.0]

# v4/v5/v6/v7/v8: 校准友好动态融合设置
# gamma < 1 会把 beta gate 收缩回 repo 原始 R 权重 prior_r=1-alpha，避免 omega 从 0.3 直接跳到 0.7+。
GAMMA_GRID_VALUES = [0.25, 0.5, 0.75, 1.0]

# v9: BayesMMRL 专用，把 R 分支 MC epistemic uncertainty 改为 percentile/rank reliability。
# score += lambda_e * (0.5 - epi_rank)，lambda_e >= 0。
# epi_rank 低于 0.5 -> 提高 R 权重；高于 0.5 -> 降低 R 权重。
USE_EPISTEMIC_RANK_RELIABILITY = True
EPISTEMIC_RELIABILITY_MODES = [
    "epistemic_rank_reliability",
    "epistemic_margin_rank_reliability",
    "epistemic_high_rank_veto",
]
LAMBDA_E_GRID_VALUES = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]

# rank reliability 不再用 raw MI 的 median/mean 作为 tau；0.5 表示验证集分位数中位线。
# high-rank veto 会把 tau_e 当作高不确定性阈值。
EPISTEMIC_TAU_MODES = ["rank_center", "rank_p90", "rank_p95"]

# 为避免 v6 那种 13 万组大网格，v9 默认使用两阶段搜索：
# 先标准 beta gate 选择 beta_u/beta_m/beta_d/bias/gamma；
# 再固定这些参数，只扫 lambda_e / tau_mode，并允许 gamma 重新选择。
EPISTEMIC_RELIABILITY_FIX_BASE_BETA = True

# 是否把完整 [N, S, C] rep MC logits stack 存进 npz。一般只需保存逐样本 epistemic 指标，False 更省空间。
SAVE_MC_REP_LOGITS_STACK = False

# beta 选择策略：先保证 val correct 数接近最优，再在候选里优先选 NLL/ECE 更好的、更不偏离 prior 的 gate。
# v5 默认用 correct_tolerance，避免浮点 accuracy 阈值把“只差 1 个样本”的校准友好候选误排除。
CORRECT_TOLERANCE = 1
CALIBRATION_ACC_EPS = 0.0025  # fallback；一般优先用 CORRECT_TOLERANCE。
MAX_OMEGA_SHIFT =  0.01        # 可设 0.15 作为 hard constraint；None 表示不强制。

# 对最终 dynamic logits 做验证集 temperature scaling；不改变 top-1，只改善置信度校准。
TEMPERATURE_GRID_VALUES = np.linspace(0.5, 5.0, 181)

# 固定 BayesMMRL MC eval 随机种子，降低重复运行波动。
EVAL_SEED = 2026

# 本版建议复用 v6/v7 full-MC epistemic 缓存；首次运行若无 npz 可改 False 重新收集。
SAVE_NPZ_CACHE = True
USE_NPZ_CACHE_IF_EXISTS = True

# 关键开关：必须从真实日志恢复配置。
REQUIRE_REAL_LOG_CONFIG = True

OUT_DIR = REPO_ROOT / "output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored_v9_epistemic_rank_reliability"
# 如果你已经跑过 v6/v7/v8 full-MC epistemic，v9 会在自己的 OUT_DIR 没有 npz 时尝试从这里读缓存。
FALLBACK_NPZ_CACHE_DIR = REPO_ROOT / "output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored_v6_fullmc_epistemic"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert REPO_ROOT.exists(), f"REPO_ROOT 不存在: {REPO_ROOT}"
assert DATA_ROOT.exists(), f"DATA_ROOT 不存在: {DATA_ROOT}"
for case in CASES:
    assert Path(case["case_root"]).expanduser().exists(), f'case_root 不存在: {case["case_root"]}'

print("REPO_ROOT =", REPO_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUT_DIR   =", OUT_DIR)
print("CASES     =", len(CASES))


REPO_ROOT = /root/autodl-tmp/MMRL
DATA_ROOT = /root/autodl-tmp/MMRL/DATASETS
OUT_DIR   = /root/autodl-tmp/MMRL/output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored_v9_epistemic_rank_reliability
CASES     = 2


In [12]:
import os
import sys
import ast
import math
import json
import random
import re
import importlib
from types import SimpleNamespace
from itertools import product

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from core.config import setup_cfg
from dassl.engine import build_trainer
from core.utils import import_optional_modules
from core.types import MethodOutputs

# 与 run.py 对齐：注册 datasets + trainer。
import_optional_modules([
    "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
    "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
    "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
    "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
])
importlib.import_module("trainers.refactor_runner")

print("torch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())


torch = 2.11.0+cu128
cuda available = True


## 1. 从真实日志恢复配置

`setup_cfg(args)` 会 merge config files 后再应用 `args.opts`。因此真实训练日志中的 `opts` 是模型配置的一部分，不能只靠路径推断。

本节会解析真实日志中的：

```text
***************
** Arguments **
***************
...
opts: [...]
...
************
** Config **
************
```

如果找不到日志或找不到 `Arguments` 区块，会直接报错，避免静默生成错误分支结果。


In [13]:
METHOD_CFG_MAP = {
    "MMRL": "configs/methods/mmrl.yaml",
    "MMRLMix": "configs/methods/mmrl_mix.yaml",
    "MMRLpp": "configs/methods/mmrlpp.yaml",
    "MMRLPP": "configs/methods/mmrlpp.yaml",
    "BayesMMRL": "configs/methods/bayesmmrl.yaml",
    "ClipAdapters": "configs/methods/clip_adapters.yaml",
    "ClipADAPTER": "configs/methods/clip_adapters.yaml",
}

PROTOCOL_CFG_MAP = {
    "B2N": "configs/protocols/b2n.yaml",
    "FS": "configs/protocols/fs.yaml",
    "CD": "configs/protocols/cd.yaml",
}

PROTOCOL_TO_SUBSAMPLE = {
    "B2N": "base",
    "FS": "all",
    "CD": "all",
}

def decode_backbone_from_dir(token: str) -> str:
    # output_refactor 中通常把 ViT-B/16 写成 ViT-B-16。
    if token.startswith("ViT-") and token.count("-") >= 2:
        head, patch = token.rsplit("-", 1)
        return f"{head}/{patch}"
    return token

def infer_case_metadata(case_root: Path):
    case_root = Path(case_root).expanduser().resolve()
    parts = case_root.parts
    if "output_refactor" in parts:
        idx = parts.index("output_refactor")
    elif "output_sweeps" in parts:
        # sweep 路径通常形如:
        # output_sweeps/.../<stage>/<Method>/<Protocol>/<phase>/<dataset>/shots_.../...
        # 向后找第一个已知 method token。
        method_indices = [i for i, p in enumerate(parts) if p in METHOD_CFG_MAP]
        if not method_indices:
            raise ValueError(f"无法从 sweep 路径解析 method: {case_root}")
        idx = method_indices[0] - 1
    else:
        raise ValueError(f"case_root 中找不到 output_refactor/output_sweeps: {case_root}")

    # 优先按 output_refactor 标准路径解析；失败时保留最少 meta，真实 cfg 由日志恢复。
    meta = {"case_root": case_root}
    try:
        if parts[idx] == "output_refactor":
            method = parts[idx + 1]
            protocol = parts[idx + 2]
            phase = parts[idx + 3]
            dataset = parts[idx + 4]
            shots_str = parts[idx + 5]
            backbone_token = parts[idx + 6]
            tag = parts[idx + 7]
            seed_dir = parts[idx + 8]
        else:
            method = parts[idx + 1]
            protocol = parts[idx + 2]
            phase = parts[idx + 3]
            dataset = parts[idx + 4]
            shots_str = parts[idx + 5]
            backbone_token = parts[idx + 6]
            tag = parts[idx + 7]
            seed_dir = parts[idx + 8]

        meta.update({
            "method": method,
            "protocol": protocol,
            "phase": phase,
            "dataset": dataset,
            "shots": int(shots_str.split("_", 1)[1]) if shots_str.startswith("shots_") else None,
            "backbone": decode_backbone_from_dir(backbone_token),
            "tag": tag,
            "seed": int(seed_dir.replace("seed", "")) if seed_dir.startswith("seed") else None,
        })
    except Exception as exc:
        meta["path_parse_warning"] = repr(exc)

    return meta

def find_real_log_file(case_root: Path, explicit_log_file=None):
    if explicit_log_file:
        p = Path(explicit_log_file).expanduser().resolve()
        if p.exists() and p.is_file():
            return p
        raise FileNotFoundError(f"显式指定的 log_file 不存在: {p}")

    case_root = Path(case_root).expanduser().resolve()
    candidates = [
        case_root / "run.log",
        case_root / "log.txt",
    ]

    # log.txt-2026-... 一般是历史复制，按 mtime 逆序尝试。
    candidates += sorted(case_root.glob("log.txt-*"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
    candidates += sorted(case_root.glob("*.log"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

    seen = set()
    unique_candidates = []
    for p in candidates:
        if str(p) not in seen:
            unique_candidates.append(p)
            seen.add(str(p))

    for p in unique_candidates:
        if p.exists() and p.is_file():
            text_head = p.read_text(encoding="utf-8", errors="ignore")[:20000]
            if "** Arguments **" in text_head and "opts:" in text_head:
                return p

    raise FileNotFoundError(
        f"找不到包含真实 Arguments/opts 的训练日志: {case_root}/run.log 或 log.txt。"
        "本 notebook 不允许只从路径猜配置。"
    )

def _parse_scalar_from_log_value(value: str):
    value = value.strip()
    if value in {"", "None"}:
        return None if value == "None" else ""
    if value in {"True", "False"}:
        return value == "True"
    # opts 必须按 Python literal 解析。
    if value.startswith("[") and value.endswith("]"):
        return ast.literal_eval(value)
    # seed/load_epoch 等单独处理；其他保留字符串，和 argparse 行为一致。
    return value

def parse_args_from_log(log_path: Path):
    text = Path(log_path).read_text(encoding="utf-8", errors="ignore")

    m = re.search(
        r"\*\* Arguments \*\*\s*\n\*+\s*\n(?P<body>.*?)(?:\n\*{4,}\s*\n\*\* Config \*\*|\nCollecting env info|\nLoading trainer:)",
        text,
        flags=re.S,
    )
    if m is None:
        raise ValueError(f"日志中找不到 Arguments 区块: {log_path}")

    arg_text = m.group("body")
    parsed = {}

    # Arguments 区块一般一行一个 "key: value"。
    for raw_line in arg_text.splitlines():
        line = raw_line.rstrip()
        if not line or ": " not in line:
            continue
        key, value = line.split(": ", 1)
        key = key.strip()
        value = value.strip()
        parsed[key] = _parse_scalar_from_log_value(value)

    if "opts" in parsed and isinstance(parsed["opts"], str):
        parsed["opts"] = ast.literal_eval(parsed["opts"])

    # 类型归一化。
    for key in ["eval_only", "no_train"]:
        if key in parsed and isinstance(parsed[key], str):
            parsed[key] = parsed[key] == "True"

    for key in ["seed", "load_epoch"]:
        if key in parsed:
            if parsed[key] in {None, ""}:
                parsed[key] = None
            else:
                parsed[key] = int(parsed[key])

    required = [
        "dataset_config_file",
        "method_config_file",
        "protocol_config_file",
        "runtime_config_file",
        "method",
        "protocol",
        "exec_mode",
        "seed",
        "trainer",
        "opts",
    ]
    missing = [k for k in required if k not in parsed]
    if missing:
        raise ValueError(f"日志 Arguments 缺少字段 {missing}: {log_path}")

    if not isinstance(parsed["opts"], list):
        raise TypeError(f"日志 opts 不是 list: {type(parsed['opts'])}, log={log_path}")

    return parsed

def make_args_from_case(case):
    case_root = Path(case["case_root"]).expanduser().resolve()
    meta = infer_case_metadata(case_root)

    log_path = find_real_log_file(case_root, case.get("log_file", None))
    logged = parse_args_from_log(log_path)

    # 注意：
    # - root 用当前机器的 DATA_ROOT，避免日志中的相对路径 DATASETS 在 notebook 环境下失效。
    # - output_dir/model_dir 用当前 case_root，确保加载当前 checkpoint。
    args = SimpleNamespace(
        root=str(DATA_ROOT),
        output_dir=str(case_root),
        dataset_config_file=str(logged["dataset_config_file"]),
        method_config_file=str(logged["method_config_file"]),
        protocol_config_file=str(logged["protocol_config_file"]),
        runtime_config_file=str(logged["runtime_config_file"]),
        exp_config=str(logged.get("exp_config", "") or ""),
        method=str(logged["method"]),
        protocol=str(logged["protocol"]),
        exec_mode=str(logged["exec_mode"]),
        seed=int(logged["seed"]) if logged.get("seed") is not None else -1,
        trainer=str(logged.get("trainer", "RefactorRunner")),
        eval_only=True,
        model_dir=str(case_root),
        load_epoch=case.get("load_epoch", None),
        no_train=True,
        opts=list(logged["opts"]),
    )

    meta.update({
        "name": case.get("name", case_root.name),
        "log_path": str(log_path),
        "opts_restored_from_log": True,
        "logged_method": logged["method"],
        "logged_protocol": logged["protocol"],
        "logged_exec_mode": logged["exec_mode"],
        "logged_seed": logged["seed"],
        "logged_opts": list(logged["opts"]),
        "logged_output_dir": logged.get("output_dir", ""),
        "logged_root": logged.get("root", ""),
    })

    if REQUIRE_REAL_LOG_CONFIG:
        assert meta["opts_restored_from_log"], "必须从真实日志恢复 opts；不能仅从路径猜配置。"

    return args, meta

def build_trainer_for_case(case):
    args, meta = make_args_from_case(case)
    cfg = setup_cfg(args)
    trainer = build_trainer(cfg)
    trainer.load_model(str(meta["case_root"]), epoch=args.load_epoch)
    trainer.set_model_mode("eval")
    return trainer, meta, args

def get_cfg_alpha(trainer):
    cfg = trainer.cfg
    if str(cfg.METHOD.NAME) == "BayesMMRL":
        return float(cfg.BAYES_MMRL.ALPHA)
    if hasattr(cfg, "MMRL") and hasattr(cfg.MMRL, "ALPHA"):
        return float(cfg.MMRL.ALPHA)
    if hasattr(trainer.method, "method_cfg") and hasattr(trainer.method.method_cfg, "ALPHA"):
        return float(trainer.method.method_cfg.ALPHA)
    return float("nan")


In [14]:
def audit_trainer_config(trainer, meta, args, case_name):
    cfg = trainer.cfg
    rows = []

    def add(key, value, expected=None):
        rows.append({
            "case_name": case_name,
            "key": key,
            "value": str(value),
            "expected_or_inferred": "" if expected is None else str(expected),
            "match": "" if expected is None else bool(str(value) == str(expected)),
        })

    add("log_path", meta.get("log_path", ""))
    add("opts_restored_from_log", meta.get("opts_restored_from_log", False), True)
    add("logged_output_dir", meta.get("logged_output_dir", ""))
    add("logged_root", meta.get("logged_root", ""))
    add("cfg.METHOD.NAME", cfg.METHOD.NAME, meta.get("logged_method", meta.get("method", "")))
    add("cfg.METHOD.EXEC_MODE", cfg.METHOD.EXEC_MODE, meta.get("logged_exec_mode", args.exec_mode))
    add("cfg.PROTOCOL.NAME", cfg.PROTOCOL.NAME, meta.get("logged_protocol", meta.get("protocol", "")))
    add("cfg.PROTOCOL.PHASE", getattr(cfg.PROTOCOL, "PHASE", None), meta.get("phase", ""))
    add("cfg.DATASET.NAME", cfg.DATASET.NAME, meta.get("dataset", ""))
    add("cfg.DATASET.NUM_SHOTS", cfg.DATASET.NUM_SHOTS, meta.get("shots", ""))
    add("cfg.DATASET.SUBSAMPLE_CLASSES", cfg.DATASET.SUBSAMPLE_CLASSES)
    add("cfg.MODEL.BACKBONE.NAME", cfg.MODEL.BACKBONE.NAME, meta.get("backbone", ""))
    add("cfg.SEED", cfg.SEED, meta.get("logged_seed", meta.get("seed", "")))
    add("cfg.OUTPUT_DIR", cfg.OUTPUT_DIR, meta["case_root"])

    if str(cfg.METHOD.NAME) == "BayesMMRL":
        keys = [
            "ALPHA", "REG_WEIGHT", "N_REP_TOKENS", "REP_LAYERS", "REP_DIM",
            "BAYES_TARGET", "N_MC_TRAIN", "N_MC_TEST",
            "EVAL_MODE", "EVAL_USE_POSTERIOR_MEAN", "EVAL_AGGREGATION",
            "USE_MEAN_MAIN_MC_REP", "KL_WARMUP_EPOCHS",
            "REP_SIGMA_MODE", "REP_PRIOR_MODE", "REP_PRIOR_STD", "REP_KL_WEIGHT",
            "PROJ_REP_SIGMA_MODE", "PROJ_REP_PRIOR_MODE", "PROJ_REP_PRIOR_STD", "PROJ_REP_KL_WEIGHT",
            "MAIN_CONSISTENCY_WEIGHT", "MAIN_CONSISTENCY_MODE",
        ]
        for k in keys:
            if hasattr(cfg.BAYES_MMRL, k):
                add(f"cfg.BAYES_MMRL.{k}", getattr(cfg.BAYES_MMRL, k))
    elif hasattr(cfg, "MMRL"):
        for k in ["ALPHA", "REG_WEIGHT", "N_REP_TOKENS", "REP_LAYERS", "REP_DIM"]:
            if hasattr(cfg.MMRL, k):
                add(f"cfg.MMRL.{k}", getattr(cfg.MMRL, k))

    add("args.dataset_config_file", args.dataset_config_file)
    add("args.method_config_file", args.method_config_file)
    add("args.protocol_config_file", args.protocol_config_file)
    add("args.runtime_config_file", args.runtime_config_file)
    add("args.exp_config", args.exp_config)
    add("args.opts", args.opts)
    add("logged_opts", meta.get("logged_opts", []))

    return pd.DataFrame(rows)


## 2. Repo-forward 分支收集

这里仍然用项目标准路径：

- `trainer.method.forward_eval(batch, eval_ctx)`
- `outputs.logits` → C/main branch
- `outputs.aux_logits["rep"]` → R/representation branch
- `outputs.aux_logits["fusion"]` → repo fusion branch
- `trainer.method.select_eval_logits(outputs, eval_ctx)` → 和 `trainer.test()` 一致的 eval 路由


In [15]:
def set_eval_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _is_bayesmmrl_trainer(trainer):
    try:
        return str(trainer.cfg.METHOD.NAME) == "BayesMMRL"
    except Exception:
        return False


def _bayes_eval_params(trainer):
    bayes_cfg = trainer.cfg.BAYES_MMRL
    n_mc = max(1, int(getattr(trainer.method, "n_mc_test", getattr(bayes_cfg, "N_MC_TEST", 1))))
    use_posterior_mean = bool(
        getattr(
            trainer.method,
            "eval_use_posterior_mean",
            getattr(bayes_cfg, "EVAL_USE_POSTERIOR_MEAN", False),
        )
    )
    aggregation = str(getattr(bayes_cfg, "EVAL_AGGREGATION", "prob_mean"))
    return n_mc, use_posterior_mean, aggregation


def _standardize_mc_logits_stack(logits_stack, batch_size, num_classes):
    """
    将 BayesMMRL 返回的 logits_rep_stack 统一为 [B, S, C]。
    当前 repo 训练路径中常见形状是 [S, B, C]，但这里做兼容处理。
    """
    if logits_stack is None:
        return None
    if not torch.is_tensor(logits_stack) or logits_stack.dim() != 3:
        return None

    if logits_stack.shape[0] == batch_size and logits_stack.shape[-1] == num_classes:
        return logits_stack
    if logits_stack.shape[1] == batch_size and logits_stack.shape[-1] == num_classes:
        return logits_stack.permute(1, 0, 2).contiguous()
    return None


def _mc_rep_uncertainty_from_logits_stack(logits_stack_bsc):
    """
    Args:
        logits_stack_bsc: [B, S, C] rep-branch MC logits.

    Returns:
        每个样本的 R 分支 MC 不确定性。
        - epistemic_r_mi: normalized mutual information = H(E[p]) - E[H(p)]
        - r_mc_pred_entropy: normalized predictive entropy H(E[p])
        - r_mc_expected_entropy: normalized expected entropy E[H(p)]
        - r_mc_prob_var_sum: sum_c Var_s[p_s(c)]
        - r_mc_logit_var_mean: mean_c Var_s[z_s(c)]
    """
    z = logits_stack_bsc.float()
    b, s, c = z.shape
    probs = torch.softmax(z, dim=-1).clamp_min(1e-12)
    p_mean = probs.mean(dim=1).clamp_min(1e-12)

    pred_entropy = -(p_mean * p_mean.log()).sum(dim=-1)
    sample_entropy = -(probs * probs.log()).sum(dim=-1)
    expected_entropy = sample_entropy.mean(dim=1)
    mi = (pred_entropy - expected_entropy).clamp_min(0.0)

    norm = float(max(math.log(c), EPS))
    prob_var_sum = probs.var(dim=1, unbiased=False).sum(dim=-1)
    logit_var_mean = z.var(dim=1, unbiased=False).mean(dim=-1)

    return {
        "epistemic_r_mi": (mi / norm).detach().cpu(),
        "r_mc_pred_entropy": (pred_entropy / norm).detach().cpu(),
        "r_mc_expected_entropy": (expected_entropy / norm).detach().cpu(),
        "r_mc_prob_var_sum": prob_var_sum.detach().cpu(),
        "r_mc_logit_var_mean": logit_var_mean.detach().cpu(),
        "r_mc_num_samples": torch.full((b,), int(s), dtype=torch.long),
    }


@torch.no_grad()
def _try_bayes_forward_eval_with_rep_mc_stack(trainer, image, label, eval_ctx):
    """
    对 BayesMMRL 走与 forward_eval 相同的 mean_main_mc_rep 路径，但保留 logits_rep_stack。
    若当前配置不是该路径或 repo 函数不可用，则返回 None，调用方回退到 method.forward_eval。
    """
    if not _is_bayesmmrl_trainer(trainer):
        return None

    method = trainer.method
    model = getattr(method, "model", None)
    if model is None or not hasattr(model, "forward_mean_main_mc_rep"):
        return None

    if hasattr(method, "_use_mean_main_mc_rep") and not bool(method._use_mean_main_mc_rep()):
        return None

    n_mc, use_posterior_mean, aggregation = _bayes_eval_params(trainer)

    out = model.forward_mean_main_mc_rep(
        image,
        num_samples=n_mc,
        use_posterior_mean_for_rep=use_posterior_mean,
        aggregation=aggregation,
    )

    z_c = out["logits_main"]
    z_r = out["logits_rep"]
    z_aux_fusion = out["logits_fusion"]

    features = {}
    if "image_features_main" in out:
        features["img"] = out["image_features_main"]
    if "text_features" in out:
        features["text"] = out["text_features"][: getattr(method, "num_classes", out["text_features"].shape[0])]

    outputs = MethodOutputs(
        logits=z_c,
        labels=label,
        aux_logits={"rep": z_r, "fusion": z_aux_fusion},
        features=features,
    )

    logits_rep_stack = _standardize_mc_logits_stack(
        out.get("logits_rep_stack", None),
        batch_size=int(label.shape[0]),
        num_classes=int(z_r.shape[-1]),
    )

    return outputs, logits_rep_stack


@torch.no_grad()
def _try_bayes_full_mc_rep_logits_stack(trainer, image, label, eval_ctx):
    """
    v6: 支持 BayesMMRL full-MC eval 路径。

    当 trainer.method._use_mean_main_mc_rep() 为 False 时，repo 的 forward_eval 只返回聚合后的
    logits/logits_rep/logits_fusion；为了计算 R 分支 epistemic uncertainty，需要额外从
    model.forward_train_samples(image, n_mc) 中取每个 MC sample 的 logits_rep。

    注意：这里不替换 repo 的 forward_eval 输出，只额外收集 MC rep logits stack 作为 gate 特征。
    因此 repo baseline / z_c / z_r / z_aux_fusion 仍然完全来自标准 forward_eval/select_eval_logits。
    """
    if not _is_bayesmmrl_trainer(trainer):
        return None

    method = trainer.method
    model = getattr(method, "model", None)
    if model is None or not hasattr(model, "forward_train_samples"):
        return None

    n_mc, use_posterior_mean, aggregation = _bayes_eval_params(trainer)
    n_mc = max(1, int(n_mc))
    if n_mc <= 1:
        return None

    try:
        sample_outputs = model.forward_train_samples(image, n_mc)
    except Exception as exc:
        # 不让 epistemic 诊断影响主 forward/eval；失败时在返回数据中体现为 epistemic_available=0。
        if bool(globals().get("PRINT_EPISTEMIC_EXTRACTION_ERRORS", False)):
            print("[warn] full-MC rep stack extraction failed:", repr(exc))
        return None

    if sample_outputs is None:
        return None

    logits_rep_list = []
    try:
        for out in sample_outputs:
            # BayesMMRL sample tuple: (logits, logits_rep, logits_fusion, image_features, text_features)
            if isinstance(out, (tuple, list)) and len(out) >= 2:
                logits_rep_list.append(out[1])
            elif isinstance(out, dict) and "logits_rep" in out:
                logits_rep_list.append(out["logits_rep"])
            else:
                return None

        if not logits_rep_list:
            return None

        logits_stack_sbc = torch.stack(logits_rep_list, dim=0)  # [S, B, C]
        return _standardize_mc_logits_stack(
            logits_stack_sbc,
            batch_size=int(label.shape[0]),
            num_classes=int(logits_stack_sbc.shape[-1]),
        )
    except Exception as exc:
        if bool(globals().get("PRINT_EPISTEMIC_EXTRACTION_ERRORS", False)):
            print("[warn] full-MC rep stack standardization failed:", repr(exc))
        return None


@torch.no_grad()
def collect_repo_forward_logits(trainer, split="test", max_batches=None, seed=2026):
    set_eval_seed(seed)
    trainer.set_model_mode("eval")

    if split == "val" and getattr(trainer, "val_loader", None) is not None:
        data_loader = trainer.val_loader
        actual_split = "val"
    else:
        data_loader = trainer.test_loader
        actual_split = "test"

    eval_ctx = trainer.executor.build_eval_context(trainer, actual_split)

    z_c_all, z_r_all, z_aux_fusion_all, z_repo_eval_all, y_all, index_all = [], [], [], [], [], []
    epi_rows = {
        "epistemic_r_mi": [],
        "r_mc_pred_entropy": [],
        "r_mc_expected_entropy": [],
        "r_mc_prob_var_sum": [],
        "r_mc_logit_var_mean": [],
        "r_mc_num_samples": [],
    }
    z_r_mc_stack_all = []
    n_seen = 0

    for batch_idx, batch in enumerate(data_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        image = batch["img"].to(trainer.device)
        label = batch["label"].to(trainer.device).long()

        bayes_eval = _try_bayes_forward_eval_with_rep_mc_stack(trainer, image, label, eval_ctx)
        if bayes_eval is None:
            outputs = trainer.method.forward_eval(
                {"img": image, "label": label},
                eval_ctx,
            )
            # v6: 如果当前 BayesMMRL 走 full-MC 路径，forward_eval 只返回聚合 logits；
            # 额外从 model.forward_train_samples(...) 取每个 MC sample 的 rep logits 计算 epistemic。
            logits_rep_stack = _try_bayes_full_mc_rep_logits_stack(
                trainer,
                image,
                label,
                eval_ctx,
            )
        else:
            outputs, logits_rep_stack = bayes_eval

        if outputs.aux_logits is None or "rep" not in outputs.aux_logits or "fusion" not in outputs.aux_logits:
            raise RuntimeError(
                "outputs.aux_logits 必须包含 'rep' 和 'fusion'。"
                f" 当前 aux_logits keys={None if outputs.aux_logits is None else list(outputs.aux_logits.keys())}"
            )

        z_c = outputs.logits
        z_r = outputs.aux_logits["rep"]
        z_aux_fusion = outputs.aux_logits["fusion"]
        z_repo_eval = trainer.method.select_eval_logits(outputs, eval_ctx)

        if not (z_c.shape == z_r.shape == z_aux_fusion.shape == z_repo_eval.shape):
            raise RuntimeError(
                f"logits shape 不一致: z_c={tuple(z_c.shape)}, z_r={tuple(z_r.shape)}, "
                f"z_aux_fusion={tuple(z_aux_fusion.shape)}, z_repo_eval={tuple(z_repo_eval.shape)}"
            )

        bsz = int(label.shape[0])
        z_c_all.append(z_c.detach().float().cpu())
        z_r_all.append(z_r.detach().float().cpu())
        z_aux_fusion_all.append(z_aux_fusion.detach().float().cpu())
        z_repo_eval_all.append(z_repo_eval.detach().float().cpu())
        y_all.append(label.detach().cpu())
        index_all.append(torch.arange(n_seen, n_seen + bsz, dtype=torch.long))
        n_seen += bsz

        if logits_rep_stack is not None:
            unc = _mc_rep_uncertainty_from_logits_stack(logits_rep_stack)
            for k in epi_rows:
                epi_rows[k].append(unc[k])
            if bool(globals().get("SAVE_MC_REP_LOGITS_STACK", False)):
                z_r_mc_stack_all.append(logits_rep_stack.detach().float().cpu())
        else:
            for k in epi_rows:
                if k == "r_mc_num_samples":
                    epi_rows[k].append(torch.zeros((bsz,), dtype=torch.long))
                else:
                    epi_rows[k].append(torch.full((bsz,), float("nan"), dtype=torch.float32))

    if not y_all:
        raise RuntimeError(f"No batches collected for split={split}")

    data = {
        "actual_split": actual_split,
        "z_c": torch.cat(z_c_all, dim=0).numpy(),
        "z_r": torch.cat(z_r_all, dim=0).numpy(),
        "z_aux_fusion": torch.cat(z_aux_fusion_all, dim=0).numpy(),
        "z_repo_eval": torch.cat(z_repo_eval_all, dim=0).numpy(),
        "y": torch.cat(y_all, dim=0).numpy(),
        "index": torch.cat(index_all, dim=0).numpy(),
    }

    for k, chunks in epi_rows.items():
        if chunks:
            data[k] = torch.cat(chunks, dim=0).numpy()

    if z_r_mc_stack_all:
        data["z_r_mc_stack"] = torch.cat(z_r_mc_stack_all, dim=0).numpy()

    return data


def cache_path_for(case_name, split, base_dir=None):
    safe = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in case_name)
    root = Path(base_dir) if base_dir is not None else OUT_DIR
    return root / f"{safe}_{split}_repo_forward_logits.npz"


def _load_npz_cache(path):
    data = dict(np.load(path, allow_pickle=True))
    data["actual_split"] = str(data["actual_split"].item() if hasattr(data["actual_split"], "item") else data["actual_split"])
    return data


def load_or_collect_case_split(trainer, case_name, split, max_batches=None, seed=2026):
    path = cache_path_for(case_name, split)
    if USE_NPZ_CACHE_IF_EXISTS and path.exists():
        return _load_npz_cache(path)

    # v7: 如果已经跑过 v6 full-MC epistemic，可以自动复用 v6 的 npz，避免重新跑 GPU forward。
    fallback_dir = globals().get("FALLBACK_NPZ_CACHE_DIR", None)
    if USE_NPZ_CACHE_IF_EXISTS and fallback_dir is not None:
        fallback_path = cache_path_for(case_name, split, base_dir=fallback_dir)
        if fallback_path.exists():
            data = _load_npz_cache(fallback_path)
            if SAVE_NPZ_CACHE:
                np.savez_compressed(path, **data)
            return data

    data = collect_repo_forward_logits(
        trainer=trainer,
        split=split,
        max_batches=max_batches,
        seed=seed,
    )
    if SAVE_NPZ_CACHE:
        np.savez_compressed(path, **data)
    return data


## 3. 指标与动态融合函数


In [16]:
EPS = 1e-12

def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    m = np.max(z, axis=1, keepdims=True)
    logsum = m + np.log(np.sum(np.exp(z - m), axis=1, keepdims=True))
    return z - logsum

def top1_acc_from_logits(z, y):
    return float(np.mean(np.argmax(z, axis=1) == y))

def n_correct_from_logits(z, y):
    return int(np.sum(np.argmax(z, axis=1) == y))

def nll_from_logits(z, y):
    logp = log_softmax_np(z)
    return float(-np.mean(logp[np.arange(len(y)), y]))

def brier_from_logits(z, y):
    p = softmax_np(z)
    onehot = np.zeros_like(p)
    onehot[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((p - onehot) ** 2, axis=1)))

def ece_from_logits(z, y, n_bins=15):
    p = softmax_np(z)
    conf = np.max(p, axis=1)
    pred = np.argmax(p, axis=1)
    correct = (pred == y).astype(np.float64)

    ece = 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == 0:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf > lo) & (conf <= hi)
        if np.any(mask):
            ece += np.mean(mask) * abs(np.mean(correct[mask]) - np.mean(conf[mask]))
    return float(ece)

def confidence_diagnostics_from_logits(z, y):
    p = softmax_np(z)
    conf = np.max(p, axis=1)
    pred = np.argmax(p, axis=1)
    correct = pred == y
    ent = entropy_np(p)

    return {
        "mean_conf": float(np.mean(conf)),
        "correct_conf": float(np.mean(conf[correct])) if np.any(correct) else float("nan"),
        "wrong_conf": float(np.mean(conf[~correct])) if np.any(~correct) else float("nan"),
        "mean_entropy": float(np.mean(ent)),
        "mean_logit_l2": float(np.mean(np.linalg.norm(z, axis=1))),
        "max_conf": float(np.max(conf)),
    }

def entropy_np(p):
    p = np.asarray(p, dtype=np.float64)
    return -np.sum(np.clip(p, EPS, 1.0) * np.log(np.clip(p, EPS, 1.0)), axis=1)

def margin_np(p):
    vals = np.sort(p, axis=1)
    if vals.shape[1] == 1:
        return vals[:, -1]
    return vals[:, -1] - vals[:, -2]

def js_divergence_np(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    m = 0.5 * (p + q)
    return 0.5 * np.sum(np.clip(p, EPS, 1.0) * (np.log(np.clip(p, EPS, 1.0)) - np.log(np.clip(m, EPS, 1.0))), axis=1) + \
           0.5 * np.sum(np.clip(q, EPS, 1.0) * (np.log(np.clip(q, EPS, 1.0)) - np.log(np.clip(m, EPS, 1.0))), axis=1)

def branch_metrics_row(case_name, split, branch_name, z, y):
    row = {
        "case_name": case_name,
        "split": split,
        "branch": branch_name,
        "n": int(len(y)),
        "n_correct": n_correct_from_logits(z, y),
        "accuracy": top1_acc_from_logits(z, y),
        "accuracy_pct": 100.0 * top1_acc_from_logits(z, y),
        "nll": nll_from_logits(z, y),
        "brier": brier_from_logits(z, y),
        "ece": ece_from_logits(z, y),
    }
    # v4: 加入置信度/尖锐度诊断。这样可以直接验证 ECE 变差到底是不是过置信/低熵/logit norm 变大。
    row.update(confidence_diagnostics_from_logits(z, y))
    return row

def get_epistemic_r_feature(data):
    """
    从 collect_repo_forward_logits 的输出中取 BayesMMRL R 分支 MC epistemic uncertainty。
    返回 normalized mutual information，shape=[N]；普通 MMRL 或未收集到时返回 None。
    """
    if data is None or "epistemic_r_mi" not in data:
        return None
    arr = np.asarray(data["epistemic_r_mi"], dtype=np.float64)
    if arr.ndim != 1 or arr.shape[0] == 0:
        return None
    finite = np.isfinite(arr)
    if not np.any(finite):
        return None
    return arr


def fit_epistemic_stats(epistemic_r):
    """
    在 beta-select split 上拟合 epistemic feature 的尺度和 rank 参考分布。

    v9:
      - raw normalized MI 仍保留为诊断；
      - scaled MI 只作辅助诊断；
      - gate 主特征改为 percentile/rank，避免 raw MI 长尾导致低不确定性奖励过弱、高不确定性惩罚过强。
    """
    if epistemic_r is None:
        return None
    x = np.asarray(epistemic_r, dtype=np.float64)
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        return None

    finite_sorted = np.sort(finite)
    p95 = float(np.percentile(finite, 95.0))
    p99 = float(np.percentile(finite, 99.0))
    mean = float(np.mean(finite))
    median = float(np.median(finite))
    scale = p95 if p95 > EPS else (p99 if p99 > EPS else (float(np.max(finite)) if np.max(finite) > EPS else 1.0))

    mean_scaled = float(np.clip(mean / max(scale, EPS), 0.0, 1.0))
    median_scaled = float(np.clip(median / max(scale, EPS), 0.0, 1.0))
    p95_scaled = float(np.clip(p95 / max(scale, EPS), 0.0, 1.0))

    # 为 rank approximation 记录一组 quantile anchor；同时在内存里保留 exact sorted ref。
    rank_probs = np.array([0.0, 0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50,
                           0.60, 0.70, 0.80, 0.90, 0.95, 0.99, 1.0], dtype=np.float64)
    rank_quantiles = np.quantile(finite_sorted, rank_probs)

    stats = {
        "epistemic_r_scale": float(scale),
        "epistemic_r_mean_ref": mean,
        "epistemic_r_median_ref": median,
        "epistemic_r_p95_ref": p95,
        "epistemic_r_p99_ref": p99,
        "epistemic_tau_mean_scaled": mean_scaled,
        "epistemic_tau_median_scaled": median_scaled,
        "epistemic_tau_p95_scaled": p95_scaled,
        # v9 rank anchors as scalar fields, safe for Excel.
        "epistemic_rank_center": 0.5,
        "epistemic_rank_p80": 0.80,
        "epistemic_rank_p90": 0.90,
        "epistemic_rank_p95": 0.95,
        "epistemic_rank_p99": 0.99,
        "epistemic_rank_ref_n": int(finite_sorted.size),
    }
    for prob, q in zip(rank_probs, rank_quantiles):
        pct = int(round(prob * 100))
        stats[f"epistemic_rank_q{pct:03d}"] = float(q)

    # 私有字段：val 搜索时用于 exact percentile rank；不会写入 beta_df。
    stats["_epistemic_ref_sorted"] = finite_sorted
    stats["_epistemic_rank_probs"] = rank_probs
    stats["_epistemic_rank_quantiles"] = rank_quantiles
    return stats



def epistemic_stats_from_beta(beta):
    keys = [
        "epistemic_r_scale",
        "epistemic_r_mean_ref",
        "epistemic_r_median_ref",
        "epistemic_r_p95_ref",
        "epistemic_r_p99_ref",
        "epistemic_tau_mean_scaled",
        "epistemic_tau_median_scaled",
        "epistemic_tau_p95_scaled",
        "epistemic_rank_center",
        "epistemic_rank_p80",
        "epistemic_rank_p90",
        "epistemic_rank_p95",
        "epistemic_rank_p99",
        "epistemic_rank_ref_n",
    ]
    # Reconstruct quantile anchors for approximate rank evaluation on test.
    keys += [f"epistemic_rank_q{pct:03d}" for pct in [0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]]

    stats = {}
    for k in keys:
        if beta is not None and k in beta and beta[k] not in ("", None):
            try:
                stats[k] = float(beta[k])
            except Exception:
                pass

    if "epistemic_r_scale" not in stats:
        return None

    # Build arrays from scalar quantile anchors if present.
    prob_pairs = []
    for pct in [0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]:
        k = f"epistemic_rank_q{pct:03d}"
        if k in stats:
            prob_pairs.append((pct / 100.0, stats[k]))
    if prob_pairs:
        stats["_epistemic_rank_probs"] = np.array([p for p, _ in prob_pairs], dtype=np.float64)
        stats["_epistemic_rank_quantiles"] = np.array([q for _, q in prob_pairs], dtype=np.float64)

    return stats



def scale_epistemic_r(epistemic_r, stats=None, n=None):
    """
    将 normalized MI 再按 val p95 缩放到大致 [0,1]，主要用于诊断。
    v9 gate 默认不再直接使用 raw scaled MI，而使用 percentile/rank。
    """
    if epistemic_r is None:
        if n is None:
            return None
        return np.zeros((int(n),), dtype=np.float64)

    x = np.asarray(epistemic_r, dtype=np.float64)
    x = np.where(np.isfinite(x), x, 0.0)

    if stats is None:
        stats = fit_epistemic_stats(x)

    scale = 1.0
    if stats is not None:
        scale = float(stats.get("epistemic_r_scale", 1.0))
    scale = max(scale, EPS)

    return np.clip(x / scale, 0.0, 1.0)


def percentile_rank_epistemic_r(epistemic_r, stats=None, n=None):
    """
    v9: 将 R 分支 MC epistemic MI 转成验证集 percentile/rank。

    返回值范围 [0,1]：
      - 0 表示低于几乎所有验证集样本，R epistemic 很低；
      - 1 表示高于几乎所有验证集样本，R epistemic 很高。

    优先使用 val 的 exact sorted reference；如果只剩 beta row 中的 quantile anchors，
    则使用分段线性近似。
    """
    if epistemic_r is None:
        if n is None:
            return None
        return np.zeros((int(n),), dtype=np.float64)

    x = np.asarray(epistemic_r, dtype=np.float64)
    x = np.where(np.isfinite(x), x, 0.0)

    if stats is None:
        stats = fit_epistemic_stats(x)

    # Exact empirical CDF / percentile rank when val sorted ref is available.
    if stats is not None and "_epistemic_ref_sorted" in stats:
        ref = np.asarray(stats["_epistemic_ref_sorted"], dtype=np.float64)
        ref = ref[np.isfinite(ref)]
        if ref.size > 0:
            # Midrank handles many tied zeros better than right-CDF.
            left = np.searchsorted(ref, x, side="left")
            right = np.searchsorted(ref, x, side="right")
            rank = (left + right) / (2.0 * ref.size)
            return np.clip(rank, 0.0, 1.0)

    # Approximate CDF from quantile anchors stored in beta row.
    if stats is not None and "_epistemic_rank_probs" in stats and "_epistemic_rank_quantiles" in stats:
        probs = np.asarray(stats["_epistemic_rank_probs"], dtype=np.float64)
        qs = np.asarray(stats["_epistemic_rank_quantiles"], dtype=np.float64)
        finite = np.isfinite(probs) & np.isfinite(qs)
        probs, qs = probs[finite], qs[finite]
        if probs.size >= 2:
            order = np.argsort(qs)
            qs, probs = qs[order], probs[order]
            # Merge duplicate quantile values by using the midpoint probability of the tied block.
            uniq_q = []
            uniq_p = []
            i = 0
            while i < qs.size:
                j = i + 1
                while j < qs.size and abs(qs[j] - qs[i]) <= 1e-15:
                    j += 1
                uniq_q.append(float(qs[i]))
                uniq_p.append(float(np.mean(probs[i:j])))
                i = j
            if len(uniq_q) >= 2:
                rank = np.interp(x, np.array(uniq_q), np.array(uniq_p), left=0.0, right=1.0)
                return np.clip(rank, 0.0, 1.0)

    # Last resort: use self-ranking, mostly for diagnostics.
    ref = np.sort(x[np.isfinite(x)])
    if ref.size == 0:
        return np.zeros_like(x, dtype=np.float64)
    left = np.searchsorted(ref, x, side="left")
    right = np.searchsorted(ref, x, side="right")
    rank = (left + right) / (2.0 * ref.size)
    return np.clip(rank, 0.0, 1.0)



def compute_dynamic_components(z_c, z_r, epistemic_r=None, epistemic_stats=None):
    p_c = softmax_np(z_c)
    p_r = softmax_np(z_r)
    c = z_c.shape[1]

    # 归一化熵：越大表示预测分布越不确定。这是 total/predictive uncertainty，不等同于 epistemic。
    u_c = entropy_np(p_c) / max(math.log(c), EPS)
    u_r = entropy_np(p_r) / max(math.log(c), EPS)

    margin_c = margin_np(p_c)
    margin_r = margin_np(p_r)

    # 归一化 JS，范围大致在 [0, 1]。
    js = js_divergence_np(p_c, p_r) / max(math.log(2.0), EPS)

    # BayesMMRL R 分支 MC epistemic uncertainty。
    # raw 是 normalized MI；scaled 保留诊断；rank 是 v9 gate 主特征。
    if epistemic_r is None:
        epistemic_raw = np.zeros((z_c.shape[0],), dtype=np.float64)
        epistemic_scaled = np.zeros((z_c.shape[0],), dtype=np.float64)
        epistemic_rank = np.zeros((z_c.shape[0],), dtype=np.float64)
        epistemic_available = False
    else:
        epistemic_raw = np.asarray(epistemic_r, dtype=np.float64)
        if epistemic_raw.shape[0] != z_c.shape[0]:
            raise ValueError(
                f"epistemic_r length mismatch: got {epistemic_raw.shape[0]}, expected {z_c.shape[0]}"
            )
        epistemic_scaled = scale_epistemic_r(epistemic_raw, stats=epistemic_stats, n=z_c.shape[0])
        epistemic_rank = percentile_rank_epistemic_r(epistemic_raw, stats=epistemic_stats, n=z_c.shape[0])
        epistemic_raw = np.where(np.isfinite(epistemic_raw), epistemic_raw, 0.0)
        epistemic_available = True

    return {
        "p_c": p_c,
        "p_r": p_r,
        "u_c": u_c,
        "u_r": u_r,
        # C 比 R 的 predictive entropy 更高时，应该提高 R 权重。
        "u_gap": u_c - u_r,
        "margin_c": margin_c,
        "margin_r": margin_r,
        # R 比 C 更 confident 时，应该提高 R 权重。
        "margin_gain": margin_r - margin_c,
        # 分歧项不预设正负，让 beta_d 在 val 上决定。
        "js": js,
        # BayesMMRL 专用：R 分支 MC epistemic uncertainty。
        "epistemic_r_raw": epistemic_raw,
        "epistemic_r_scaled": epistemic_scaled,
        "epistemic_r_rank": epistemic_rank,
        "epistemic_available": epistemic_available,
    }



def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))

def logit_scalar_np(p):
    p = float(np.clip(p, 1e-6, 1.0 - 1e-6))
    return math.log(p / (1.0 - p))

def dynamic_weights_no_beta(z_c, z_r, prior_r=0.5):
    """
    无 beta 规则版：以 repo 原始 R 权重 prior_r=1-alpha 为中心做轻微动态调整。
    """
    comp = compute_dynamic_components(z_c, z_r)
    base = logit_scalar_np(prior_r)
    score = base + comp["u_gap"] + comp["margin_gain"]
    omega = sigmoid_np(score)
    return omega, comp

def epistemic_tau_from_stats(stats=None, tau_mode="rank_center", default=0.5):
    """
    v9: tau_e 主要用于 rank-based modes。
    - rank_center: 0.5，低于中位分位奖励 R，高于中位分位惩罚 R；
    - rank_p80/p90/p95: high-rank veto 的阈值；
    - 兼容 v8 的 median/mean/p95 scaled tau。
    """
    if stats is None:
        return float(default)

    mode = str(tau_mode or "rank_center").lower()

    rank_map = {
        "rank_center": "epistemic_rank_center",
        "rank_median": "epistemic_rank_center",
        "rank_p80": "epistemic_rank_p80",
        "rank_p90": "epistemic_rank_p90",
        "rank_p95": "epistemic_rank_p95",
        "rank_p99": "epistemic_rank_p99",
    }
    if mode in rank_map:
        return float(np.clip(stats.get(rank_map[mode], 0.5), 0.0, 1.0))

    key_map = {
        "median": "epistemic_tau_median_scaled",
        "mean": "epistemic_tau_mean_scaled",
        "p95": "epistemic_tau_p95_scaled",
    }
    if mode in key_map and key_map[mode] in stats:
        return float(np.clip(stats[key_map[mode]], 0.0, 1.0))

    # 兼容旧 stats：没有 scaled tau 时用 raw / scale 现算。
    scale = float(stats.get("epistemic_r_scale", 1.0))
    scale = max(scale, EPS)
    if mode == "median" and "epistemic_r_median_ref" in stats:
        return float(np.clip(float(stats["epistemic_r_median_ref"]) / scale, 0.0, 1.0))
    if mode == "mean" and "epistemic_r_mean_ref" in stats:
        return float(np.clip(float(stats["epistemic_r_mean_ref"]) / scale, 0.0, 1.0))
    if mode == "p95" and "epistemic_r_p95_ref" in stats:
        return float(np.clip(float(stats["epistemic_r_p95_ref"]) / scale, 0.0, 1.0))

    try:
        return float(np.clip(float(tau_mode), 0.0, 1.0))
    except Exception:
        return float(default)



def epistemic_reliability_signal(comp, mode="free", tau_e=0.5):
    """
    v9: 将 R 分支 MC epistemic uncertainty 变成 percentile/rank reliability 信号。

    主要信号：
        reliability = 0.5 - epi_rank

    - epi_rank < 0.5: reliability > 0，低认知不确定性，提高 R 权重；
    - epi_rank > 0.5: reliability < 0，高认知不确定性，降低 R 权重；
    - 相比 raw scaled MI，rank reliability 对长尾分布更稳，奖励/惩罚幅度对称。

    Modes:
      - free/none: 不使用 epistemic；
      - epistemic_rank_reliability: 直接使用 rank reliability；
      - epistemic_margin_rank_reliability: 低 MI 且 R margin 强才奖励；高 MI 且 R margin 弱才惩罚；
      - epistemic_high_rank_veto: 只在 epi_rank > tau_e 时惩罚，不奖励低 epistemic；
      - 兼容旧 v8/v7 modes。
    """
    mode = str(mode or "free")
    e_scaled = np.asarray(comp.get("epistemic_r_scaled", 0.0), dtype=np.float64)
    e_rank = np.asarray(comp.get("epistemic_r_rank", e_scaled), dtype=np.float64)
    tau = float(np.clip(tau_e, 0.0, 1.0))
    rank_centered = 0.5 - e_rank

    if mode in {"", "none", "free", "epistemic_free"}:
        return np.zeros_like(e_rank, dtype=np.float64)

    if mode in {"epistemic_rank_reliability", "rank_reliability", "percentile_rank_reliability"}:
        return rank_centered

    if mode in {"epistemic_margin_rank_reliability", "margin_rank_reliability"}:
        margin_gain = np.asarray(comp.get("margin_gain", 0.0), dtype=np.float64)
        positive = np.maximum(rank_centered, 0.0)
        negative = np.minimum(rank_centered, 0.0)
        # 只在 R margin 更强时奖励低 epistemic；只在 R margin 更弱时惩罚高 epistemic。
        return positive * (margin_gain > 0).astype(np.float64) + negative * (margin_gain < 0).astype(np.float64)

    if mode in {"epistemic_high_rank_veto", "high_rank_veto", "rank_veto"}:
        # 只对最高 epistemic 分位做 veto。tau_e 通常为 0.9/0.95。
        return -np.maximum(0.0, e_rank - tau)

    # Backward-compatible v8 centered reliability on raw scaled MI.
    if mode in {"epistemic_centered_reliability", "centered_reliability", "reliability"}:
        return tau - e_scaled

    if mode in {"epistemic_margin_centered_reliability", "margin_centered_reliability", "margin_reliability"}:
        margin_gain = np.asarray(comp.get("margin_gain", 0.0), dtype=np.float64)
        return (tau - e_scaled) * np.maximum(0.0, margin_gain)

    # Backward-compatible v7 risk penalty as negative reliability.
    if mode == "epistemic_risk":
        return -e_scaled
    if mode in {"epistemic_cond_margin_risk", "conditional_margin_risk", "cond_margin"}:
        margin_gain = np.asarray(comp.get("margin_gain", 0.0), dtype=np.float64)
        return -(e_scaled * np.maximum(0.0, -margin_gain))

    raise ValueError(f"Unsupported epistemic_penalty_mode / reliability mode: {mode}")



def dynamic_weights_beta(
    z_c,
    z_r,
    beta_u=1.0,
    beta_m=1.0,
    beta_d=0.0,
    bias=0.0,
    prior_r=0.5,
    beta_e=0.0,
    lambda_e=0.0,
    epistemic_penalty_mode="free",
    epistemic_tau_mode="median",
    tau_e=None,
    epistemic_r=None,
    epistemic_stats=None,
):
    """
    有 beta 版：在原始固定融合权重 prior_r=1-alpha 的 logit 上学习 residual gate。

    beta_u/beta_m/beta_d 全为 0 且 bias=0 时：omega_r == prior_r，严格退化到 repo 原始 alpha 融合。

    v9 改动：
      - 不再直接使用 raw scaled MI；
      - 使用 `+ lambda_e * (0.5 - epistemic_rank)` 的 percentile/rank reliability；
      - lambda_e >= 0 时，低分位 epistemic 提高 R 权重，高分位 epistemic 降低 R 权重。
    """
    comp = compute_dynamic_components(
        z_c,
        z_r,
        epistemic_r=epistemic_r,
        epistemic_stats=epistemic_stats,
    )
    base = logit_scalar_np(prior_r)
    if tau_e is None:
        tau_e = epistemic_tau_from_stats(epistemic_stats, tau_mode=epistemic_tau_mode, default=0.5)
    tau_e = float(np.clip(tau_e, 0.0, 1.0))

    reliability = epistemic_reliability_signal(comp, mode=epistemic_penalty_mode, tau_e=tau_e)
    score = (
        base
        + bias
        + beta_u * comp["u_gap"]
        + beta_m * comp["margin_gain"]
        + beta_d * comp["js"]
        # beta_e 保留兼容旧实验，v8 搜索默认不使用。
        + beta_e * comp["epistemic_r_scaled"]
        + float(lambda_e) * reliability
    )
    omega = sigmoid_np(score)
    comp["epistemic_reliability"] = reliability
    comp["epistemic_risk"] = -reliability
    comp["epistemic_penalty_mode"] = epistemic_penalty_mode
    comp["epistemic_tau_e"] = tau_e
    comp["epistemic_tau_mode"] = epistemic_tau_mode
    return omega, comp

def shrink_omega_to_prior(omega, prior_r=0.5, gamma=1.0):
    """
    v4 核心改进 1：限制动态 gate 偏离 repo 原始 prior 的幅度。
      gamma=1.0: 保留原始 beta gate；
      gamma=0.5: 只走一半动态调整；
      gamma=0.25: 更保守，通常更利于 ECE/NLL。
    """
    prior_r = float(np.clip(prior_r, 1e-6, 1.0 - 1e-6))
    gamma = float(np.clip(gamma, 0.0, 1.0))
    omega = np.asarray(omega, dtype=np.float64)
    return prior_r + gamma * (omega - prior_r)

def fuse_dynamic_logits(z_c, z_r, omega_r):
    omega_r = np.asarray(omega_r, dtype=np.float64).reshape(-1, 1)
    return (1.0 - omega_r) * z_c + omega_r * z_r

def apply_temperature(z, temperature=1.0):
    temperature = max(float(temperature), 1e-6)
    return np.asarray(z, dtype=np.float64) / temperature

def find_best_temperature_on_split(z, y, grid=None, metric="nll"):
    """
    v4 核心改进 2：验证集温度缩放。T 不改变 top-1，只改变 softmax 置信度。
    默认按 NLL 选 T，通常比直接按 ECE 更稳定。
    """
    if grid is None:
        grid = np.linspace(0.5, 5.0, 181)

    best_T = None
    best_score = None
    rows = []

    for T in grid:
        z_t = apply_temperature(z, T)
        row = {
            "temperature": float(T),
            "accuracy": top1_acc_from_logits(z_t, y),
            "nll": nll_from_logits(z_t, y),
            "brier": brier_from_logits(z_t, y),
            "ece": ece_from_logits(z_t, y),
        }
        rows.append(row)

        if metric == "nll":
            score = row["nll"]
        elif metric == "ece":
            score = row["ece"]
        elif metric == "brier":
            score = row["brier"]
        else:
            raise ValueError(f"Unsupported temperature metric: {metric}")

        if best_score is None or score < best_score:
            best_score = score
            best_T = float(T)

    return best_T, best_score, pd.DataFrame(rows)


def summarize_epistemic_data(data):
    epi = get_epistemic_r_feature(data)
    if epi is None:
        return {
            "epistemic_available": False,
            "epistemic_r_mi_mean": "",
            "epistemic_r_mi_std": "",
            "epistemic_r_mi_p95": "",
            "r_mc_num_samples_median": "",
        }

    finite = epi[np.isfinite(epi)]
    n_mc = np.asarray(data.get("r_mc_num_samples", []))
    n_mc_finite = n_mc[np.isfinite(n_mc)] if n_mc.size else np.array([])
    return {
        "epistemic_available": True,
        "epistemic_r_mi_mean": float(np.mean(finite)) if finite.size else "",
        "epistemic_r_mi_std": float(np.std(finite)) if finite.size else "",
        "epistemic_r_mi_p95": float(np.percentile(finite, 95.0)) if finite.size else "",
        "r_mc_pred_entropy_mean": float(np.nanmean(data.get("r_mc_pred_entropy", np.nan))),
        "r_mc_expected_entropy_mean": float(np.nanmean(data.get("r_mc_expected_entropy", np.nan))),
        "r_mc_prob_var_sum_mean": float(np.nanmean(data.get("r_mc_prob_var_sum", np.nan))),
        "r_mc_logit_var_mean": float(np.nanmean(data.get("r_mc_logit_var_mean", np.nan))),
        "r_mc_num_samples_median": float(np.median(n_mc_finite)) if n_mc_finite.size else "",
    }


def summarize_logits_matrix(prefix, z):
    return {
        f"{prefix}_shape": str(tuple(z.shape)),
        f"{prefix}_mean": float(np.mean(z)),
        f"{prefix}_std": float(np.std(z)),
        f"{prefix}_min": float(np.min(z)),
        f"{prefix}_max": float(np.max(z)),
        f"{prefix}_mean_l2": float(np.mean(np.linalg.norm(z, axis=1))),
    }


In [17]:
def _prior_r_from_alpha(alpha):
    if alpha is None or not np.isfinite(alpha):
        return 0.5
    return float(np.clip(1.0 - float(alpha), 1e-6, 1.0 - 1e-6))


def _beta_params_from_row(beta, default_prior_r):
    tau_e = beta.get("tau_e", None)
    if tau_e in ("", None):
        tau_e = None
    else:
        tau_e = float(tau_e)

    return {
        "beta_u": float(beta.get("beta_u", 0.0)),
        "beta_m": float(beta.get("beta_m", 0.0)),
        "beta_d": float(beta.get("beta_d", 0.0)),
        "bias": float(beta.get("bias", 0.0)),
        "prior_r": float(beta.get("prior_r", default_prior_r)),
        # v6 旧自由项；v8 默认设为 0，只保留兼容。
        "beta_e": float(beta.get("beta_e", 0.0)),
        # v8 centered reliability。
        "lambda_e": float(beta.get("lambda_e", 0.0)),
        "epistemic_penalty_mode": str(beta.get("epistemic_penalty_mode", "free") or "free"),
        "epistemic_tau_mode": str(beta.get("epistemic_tau_mode", "rank_center") or "rank_center"),
        "tau_e": tau_e,
    }


def dynamic_logits_from_beta(
    z_c,
    z_r,
    beta,
    alpha=None,
    apply_temp=False,
    epistemic_r=None,
    epistemic_stats=None,
):
    """
    根据一行 beta 搜索结果生成 dynamic logits。
    v4: 支持 gamma shrink 和 temperature scaling。
    v8: 支持 +lambda_e*(tau_e-epistemic_r_scaled) 的 centered reliability。
    """
    prior_r = _prior_r_from_alpha(alpha)
    beta_params = _beta_params_from_row(beta, prior_r)
    gamma = float(beta.get("gamma", 1.0))
    temperature = float(beta.get("temperature", 1.0))

    if epistemic_stats is None:
        epistemic_stats = epistemic_stats_from_beta(beta)

    omega_raw, comp = dynamic_weights_beta(
        z_c,
        z_r,
        **beta_params,
        epistemic_r=epistemic_r,
        epistemic_stats=epistemic_stats,
    )
    omega = shrink_omega_to_prior(omega_raw, prior_r=beta_params["prior_r"], gamma=gamma)
    z_dyn = fuse_dynamic_logits(z_c, z_r, omega)
    if apply_temp:
        z_dyn = apply_temperature(z_dyn, temperature)
    return z_dyn, omega, omega_raw, comp


def _update_beta_row_common(row, beta, omega, omega_raw, beta_params):
    gamma = float(beta.get("gamma", 1.0))
    temperature = float(beta.get("temperature", 1.0))
    row.update({
        "omega_mean": float(np.mean(omega)),
        "omega_std": float(np.std(omega)),
        "omega_min": float(np.min(omega)),
        "omega_max": float(np.max(omega)),
        "omega_raw_mean": float(np.mean(omega_raw)),
        "omega_raw_std": float(np.std(omega_raw)),
        "omega_abs_shift_from_prior": float(abs(np.mean(omega) - beta_params["prior_r"])),
        **beta_params,
        "gamma": gamma,
        "selected_on_split": beta.get("split", ""),
        "selected_val_accuracy": beta.get("accuracy", ""),
        "selected_val_nll": beta.get("nll", ""),
        "selected_val_ece": beta.get("ece", ""),
        "selected_val_n_correct": beta.get("n_correct", ""),
        "selection_policy": beta.get("selection_policy", ""),
        "search_mode": beta.get("search_mode", ""),
        "epistemic_available": beta.get("epistemic_available", False),
        "epistemic_r_scale": beta.get("epistemic_r_scale", ""),
        "epistemic_r_mean_ref": beta.get("epistemic_r_mean_ref", ""),
        "epistemic_r_p95_ref": beta.get("epistemic_r_p95_ref", ""),
        "epistemic_tau_mode": beta.get("epistemic_tau_mode", ""),
        "tau_e": beta.get("tau_e", ""),
        "epistemic_tau_median_scaled": beta.get("epistemic_tau_median_scaled", ""),
        "epistemic_tau_mean_scaled": beta.get("epistemic_tau_mean_scaled", ""),
        "epistemic_rank_center": beta.get("epistemic_rank_center", ""),
        "epistemic_rank_p80": beta.get("epistemic_rank_p80", ""),
        "epistemic_rank_p90": beta.get("epistemic_rank_p90", ""),
        "epistemic_rank_p95": beta.get("epistemic_rank_p95", ""),
        "lambda_e": beta.get("lambda_e", ""),
        "epistemic_penalty_mode": beta.get("epistemic_penalty_mode", ""),
        "epistemic_reliability_mean": beta.get("epistemic_reliability_mean", ""),
        "epistemic_reliability_std": beta.get("epistemic_reliability_std", ""),
        "epistemic_risk_mean": beta.get("epistemic_risk_mean", ""),
        "epistemic_risk_std": beta.get("epistemic_risk_std", ""),
    })
    return row


def _evaluate_one_beta_variant(case_name, split, data, beta, alpha, branch_name, apply_temp=False):
    z_c = data["z_c"]
    z_r = data["z_r"]
    y = data["y"]
    prior_r = _prior_r_from_alpha(alpha)
    epistemic_r = get_epistemic_r_feature(data)
    epistemic_stats = epistemic_stats_from_beta(beta)

    z_dyn, omega, omega_raw, comp = dynamic_logits_from_beta(
        z_c,
        z_r,
        beta,
        alpha=alpha,
        apply_temp=apply_temp,
        epistemic_r=epistemic_r,
        epistemic_stats=epistemic_stats,
    )
    row = branch_metrics_row(case_name, split, branch_name, z_dyn, y)
    beta_params = _beta_params_from_row(beta, prior_r)
    row = _update_beta_row_common(row, beta, omega, omega_raw, beta_params)
    row["temperature"] = float(beta.get("temperature", 1.0)) if apply_temp else ""
    if apply_temp:
        row["selected_val_temp_nll"] = beta.get("temp_nll", "")
        row["selected_val_temp_ece"] = beta.get("temp_ece", "")
    return row


def evaluate_dynamic_variants(case_name, split, data, beta=None, beta_epistemic=None, alpha=None):
    y = data["y"]
    z_c = data["z_c"]
    z_r = data["z_r"]
    prior_r = _prior_r_from_alpha(alpha)

    rows = []

    # 1) 无 beta 版：稳定、无验证集超参，作为保守动态 baseline。
    omega0, comp0 = dynamic_weights_no_beta(z_c, z_r, prior_r=prior_r)
    z_dyn0 = fuse_dynamic_logits(z_c, z_r, omega0)
    row0 = branch_metrics_row(case_name, split, "dynamic_no_beta_prior_centered", z_dyn0, y)
    row0.update({
        "omega_mean": float(np.mean(omega0)),
        "omega_std": float(np.std(omega0)),
        "omega_min": float(np.min(omega0)),
        "omega_max": float(np.max(omega0)),
        "prior_r": prior_r,
        "beta_u": "",
        "beta_m": "",
        "beta_d": "",
        "beta_e": "",
        "bias": "",
        "gamma": "",
        "temperature": "",
        "search_mode": "",
    })
    rows.append(row0)

    if beta is not None:
        # 2) v4 beta + gamma shrink，未温度缩放。
        rows.append(_evaluate_one_beta_variant(
            case_name,
            split,
            data,
            beta,
            alpha,
            "dynamic_beta_shrink_selected_on_val",
            apply_temp=False,
        ))

        # 3) v4 beta + gamma shrink + temperature scaling。
        temperature = float(beta.get("temperature", 1.0))
        if np.isfinite(temperature) and abs(temperature - 1.0) > 1e-12:
            rows.append(_evaluate_one_beta_variant(
                case_name,
                split,
                data,
                beta,
                alpha,
                "dynamic_beta_shrink_temp_scaled",
                apply_temp=True,
            ))

    if beta_epistemic is not None:
        # v7: 可能有多个 R-risk penalty 变体。
        beta_list = beta_epistemic if isinstance(beta_epistemic, (list, tuple)) else [beta_epistemic]
        for beta_epi in beta_list:
            mode = str(beta_epi.get("epistemic_penalty_mode", "epistemic_risk"))
            suffix = mode.replace("epistemic_", "")
            rows.append(_evaluate_one_beta_variant(
                case_name,
                split,
                data,
                beta_epi,
                alpha,
                f"dynamic_beta_epistemic_{suffix}_selected_on_val",
                apply_temp=False,
            ))

            temperature = float(beta_epi.get("temperature", 1.0))
            if np.isfinite(temperature) and abs(temperature - 1.0) > 1e-12:
                rows.append(_evaluate_one_beta_variant(
                    case_name,
                    split,
                    data,
                    beta_epi,
                    alpha,
                    f"dynamic_beta_epistemic_{suffix}_temp_scaled",
                    apply_temp=True,
                ))

    return rows


def _calibration_friendly_select(beta_df, acc_eps=None, correct_tolerance=None, max_omega_shift=None):
    """
    v5: 不再只按 val accuracy 选 beta。
    选择逻辑：
      1. 找到 val n_correct 的最高值；
      2. 保留 n_correct 在最高值 correct_tolerance 以内的候选；
      3. 可选 hard constraint: omega_mean 不偏离 prior 太多；
      4. 在这些候选中按 NLL、ECE、omega 偏离 prior 的幅度排序。
    """
    if acc_eps is None:
        acc_eps = float(globals().get("CALIBRATION_ACC_EPS", 0.0025))
    if correct_tolerance is None:
        correct_tolerance = globals().get("CORRECT_TOLERANCE", None)
    if max_omega_shift is None:
        max_omega_shift = globals().get("MAX_OMEGA_SHIFT", None)

    df = beta_df.copy()
    df["omega_abs_shift_from_prior"] = np.abs(df["omega_mean"] - df["prior_r"])

    if correct_tolerance is not None:
        best_correct = int(df["n_correct"].max())
        df["within_selection_tolerance"] = df["n_correct"] >= (best_correct - int(correct_tolerance))
        policy_head = f"n_correct within {int(correct_tolerance)} of best val correct"
    else:
        best_acc = float(df["accuracy"].max())
        df["within_selection_tolerance"] = df["accuracy"] >= (best_acc - float(acc_eps))
        policy_head = f"accuracy within {acc_eps} of best val acc"

    candidates = df[df["within_selection_tolerance"]].copy()
    if candidates.empty:
        candidates = df.copy()

    if max_omega_shift not in (None, ""):
        try:
            max_omega_shift = float(max_omega_shift)
            constrained = candidates[candidates["omega_abs_shift_from_prior"] <= max_omega_shift].copy()
            if not constrained.empty:
                candidates = constrained
                policy_head += f", omega_shift <= {max_omega_shift}"
        except Exception:
            pass

    candidates = candidates.sort_values(
        by=["nll", "ece", "omega_abs_shift_from_prior", "omega_std"],
        ascending=[True, True, True, True],
    )
    best = candidates.iloc[0].to_dict()
    best["selection_policy"] = (
        f"{policy_head}, then min nll/ece/omega_shift"
    )
    return best, df


def _search_mode_name(use_epistemic, epistemic_penalty_mode="free"):
    if not use_epistemic:
        return "standard"
    mode = str(epistemic_penalty_mode or "epistemic_risk")
    return mode


def grid_search_beta_on_split(
    case_name,
    data_val,
    grid_values,
    alpha=None,
    use_epistemic=False,
    beta_e_values=None,
    lambda_e_values=None,
    epistemic_tau_modes=None,
    epistemic_penalty_mode="free",
    base_beta=None,
):
    z_c = data_val["z_c"]
    z_r = data_val["z_r"]
    y = data_val["y"]
    prior_r = _prior_r_from_alpha(alpha)

    search_mode = _search_mode_name(use_epistemic, epistemic_penalty_mode)
    epistemic_r = get_epistemic_r_feature(data_val) if use_epistemic else None
    epistemic_available = epistemic_r is not None
    epistemic_stats = fit_epistemic_stats(epistemic_r) if epistemic_available else None

    if use_epistemic and not epistemic_available:
        raise RuntimeError(
            f"{case_name}: use_epistemic=True 但 data_val 没有有效 epistemic_r_mi。"
        )

    # v9: epistemic 不再作为 raw MI 搜索，而是 lambda_e >= 0 的 percentile/rank reliability。
    if lambda_e_values is None:
        lambda_e_values = globals().get("LAMBDA_E_GRID_VALUES", [0.0])
    lambda_e_grid = list(lambda_e_values) if use_epistemic else [0.0]

    if epistemic_tau_modes is None:
        epistemic_tau_modes = globals().get("EPISTEMIC_TAU_MODES", ["median"])
    tau_mode_grid = list(epistemic_tau_modes) if use_epistemic else ["none"]

    # 保留兼容：只有 free 模式才允许 beta_e。
    if beta_e_values is None:
        beta_e_values = globals().get("BETA_E_GRID_VALUES", [0.0])
    beta_e_grid = list(beta_e_values) if (use_epistemic and str(epistemic_penalty_mode) == "free") else [0.0]

    rows = []
    bias_grid = [-2.0, -1.0, 0.0, 1.0, 2.0]
    gamma_grid = list(globals().get("GAMMA_GRID_VALUES", [1.0]))

    # v9 默认两阶段：rank reliability 以标准 gate 的 best_beta 为 base，只扫 lambda_e / tau_mode / gamma。
    fix_base = bool(globals().get("EPISTEMIC_RELIABILITY_FIX_BASE_BETA", True))
    if use_epistemic and base_beta is not None and fix_base and str(epistemic_penalty_mode) != "free":
        beta_u_grid = [float(base_beta.get("beta_u", 0.0))]
        beta_m_grid = [float(base_beta.get("beta_m", 0.0))]
        beta_d_grid = [float(base_beta.get("beta_d", 0.0))]
        bias_grid = [float(base_beta.get("bias", 0.0))]
        # 允许 gamma 重新选择，也可固定为 base_beta["gamma"]，这里保持可调。
    else:
        beta_u_grid = list(grid_values)
        beta_m_grid = list(grid_values)
        beta_d_grid = list(grid_values)

    for beta_u, beta_m, beta_d, beta_e, lambda_e, tau_mode, bias, gamma in product(
        beta_u_grid,
        beta_m_grid,
        beta_d_grid,
        beta_e_grid,
        lambda_e_grid,
        tau_mode_grid,
        bias_grid,
        gamma_grid,
    ):
        beta = {
            "beta_u": float(beta_u),
            "beta_m": float(beta_m),
            "beta_d": float(beta_d),
            "beta_e": float(beta_e),
            "lambda_e": float(lambda_e),
            "epistemic_penalty_mode": str(epistemic_penalty_mode if use_epistemic else "free"),
            "epistemic_tau_mode": str(tau_mode),
            "bias": float(bias),
            "prior_r": float(prior_r),
            "gamma": float(gamma),
        }
        if epistemic_stats is not None:
            # Copy only public scalar stats into beta rows. Private arrays are used only during val search.
            for _k, _v in epistemic_stats.items():
                if str(_k).startswith("_"):
                    continue
                if np.isscalar(_v):
                    beta[_k] = _v
            beta["tau_e"] = epistemic_tau_from_stats(epistemic_stats, tau_mode=tau_mode, default=0.5)
        else:
            beta["tau_e"] = ""

        omega_raw, comp = dynamic_weights_beta(
            z_c,
            z_r,
            beta_u=beta["beta_u"],
            beta_m=beta["beta_m"],
            beta_d=beta["beta_d"],
            beta_e=beta["beta_e"],
            lambda_e=beta["lambda_e"],
            epistemic_penalty_mode=beta["epistemic_penalty_mode"],
            epistemic_tau_mode=beta["epistemic_tau_mode"],
            tau_e=(None if beta["tau_e"] == "" else beta["tau_e"]),
            bias=beta["bias"],
            prior_r=beta["prior_r"],
            epistemic_r=epistemic_r,
            epistemic_stats=epistemic_stats,
        )
        omega = shrink_omega_to_prior(omega_raw, prior_r=prior_r, gamma=gamma)
        z_dyn = fuse_dynamic_logits(z_c, z_r, omega)

        row = {
            "case_name": case_name,
            "split": data_val["actual_split"],
            "search_mode": search_mode,
            "epistemic_penalty_mode": beta["epistemic_penalty_mode"],
            "epistemic_available": bool(epistemic_available),
            **beta,
            "accuracy": top1_acc_from_logits(z_dyn, y),
            "accuracy_pct": 100.0 * top1_acc_from_logits(z_dyn, y),
            "n_correct": n_correct_from_logits(z_dyn, y),
            "nll": nll_from_logits(z_dyn, y),
            "brier": brier_from_logits(z_dyn, y),
            "ece": ece_from_logits(z_dyn, y),
            "omega_mean": float(np.mean(omega)),
            "omega_std": float(np.std(omega)),
            "omega_min": float(np.min(omega)),
            "omega_max": float(np.max(omega)),
            "omega_raw_mean": float(np.mean(omega_raw)),
            "omega_raw_std": float(np.std(omega_raw)),
            "epistemic_r_scaled_mean": float(np.mean(comp["epistemic_r_scaled"])),
            "epistemic_r_scaled_std": float(np.std(comp["epistemic_r_scaled"])),
            "epistemic_r_rank_mean": float(np.mean(comp.get("epistemic_r_rank", np.zeros_like(omega)))),
            "epistemic_r_rank_std": float(np.std(comp.get("epistemic_r_rank", np.zeros_like(omega)))),
            "epistemic_r_raw_mean": float(np.mean(comp["epistemic_r_raw"])),
            "epistemic_r_raw_std": float(np.std(comp["epistemic_r_raw"])),
            "epistemic_tau_e": float(comp.get("epistemic_tau_e", np.nan)),
            "epistemic_tau_mode": str(comp.get("epistemic_tau_mode", "")),
            "epistemic_reliability_mean": float(np.mean(comp.get("epistemic_reliability", np.zeros_like(omega)))),
            "epistemic_reliability_std": float(np.std(comp.get("epistemic_reliability", np.zeros_like(omega)))),
            "epistemic_risk_mean": float(np.mean(comp.get("epistemic_risk", np.zeros_like(omega)))),
            "epistemic_risk_std": float(np.std(comp.get("epistemic_risk", np.zeros_like(omega)))),
        }
        row["omega_abs_shift_from_prior"] = float(abs(row["omega_mean"] - prior_r))
        rows.append(row)

    beta_df = pd.DataFrame(rows)

    # 校准友好选 beta：correct 近似最优，再选 NLL/ECE/omega_shift。
    best, beta_df = _calibration_friendly_select(beta_df)
    best["search_mode"] = search_mode
    best["epistemic_penalty_mode"] = str(epistemic_penalty_mode if use_epistemic else "free")

    # 对选中的 beta 在 val 上拟合 temperature。T 不改变 top-1，只改善 calibration。
    z_best_val, omega_best, omega_raw_best, _ = dynamic_logits_from_beta(
        z_c,
        z_r,
        best,
        alpha=alpha,
        apply_temp=False,
        epistemic_r=epistemic_r,
        epistemic_stats=epistemic_stats,
    )
    temp_grid = globals().get("TEMPERATURE_GRID_VALUES", np.linspace(0.5, 5.0, 181))
    best_T, best_T_score, temp_df = find_best_temperature_on_split(
        z_best_val, y, grid=temp_grid, metric="nll"
    )
    z_best_val_t = apply_temperature(z_best_val, best_T)

    best["temperature"] = float(best_T)
    best["temp_metric"] = "nll"
    best["temp_nll"] = nll_from_logits(z_best_val_t, y)
    best["temp_brier"] = brier_from_logits(z_best_val_t, y)
    best["temp_ece"] = ece_from_logits(z_best_val_t, y)
    best["temp_accuracy"] = top1_acc_from_logits(z_best_val_t, y)

    if "within_selection_tolerance" not in beta_df.columns:
        beta_df["within_selection_tolerance"] = False

    beta_df = beta_df.sort_values(
        ["within_selection_tolerance", "accuracy", "nll", "ece", "omega_abs_shift_from_prior"],
        ascending=[False, False, True, True, True],
    )

    return best, beta_df


## 4. 主流程：构建 trainer、收集 logits、生成报告


In [18]:
all_accuracy_rows = []
all_sanity_rows = []
all_config_audit_rows = []
all_beta_top_rows = []
all_full_diag_rows = []
all_transition_rows = []
all_changed_rows = []
case_results = {}

for case in CASES:
    case_name = case["name"]
    print("\n" + "=" * 80)
    print("CASE:", case_name)
    print("=" * 80)

    trainer, meta, args = build_trainer_for_case(case)
    print("restored log:", meta.get("log_path"))
    print("restored opts count:", len(args.opts))

    audit_df = audit_trainer_config(trainer, meta, args, case_name)
    all_config_audit_rows.append(audit_df)

    test_data = load_or_collect_case_split(
        trainer,
        case_name,
        split=TEST_SPLIT,
        max_batches=MAX_BATCHES,
        seed=EVAL_SEED,
    )
    case_results[(case_name, "test")] = test_data

    # 尽量收集 val 以选择 beta；如果 val 不存在，会回退 test，这会在 actual_split 中显示出来。
    val_data = load_or_collect_case_split(
        trainer,
        case_name,
        split=BETA_SELECT_SPLIT,
        max_batches=MAX_BATCHES,
        seed=EVAL_SEED,
    )
    case_results[(case_name, "val_or_test")] = val_data

    for split_name, data in [("test", test_data), ("beta_select", val_data)]:
        y = data["y"]
        actual_split = data["actual_split"]

        all_accuracy_rows.extend([
            branch_metrics_row(case_name, actual_split, "C_only_outputs_logits", data["z_c"], y),
            branch_metrics_row(case_name, actual_split, "R_only_aux_logits_rep", data["z_r"], y),
            branch_metrics_row(case_name, actual_split, "repo_aux_logits_fusion", data["z_aux_fusion"], y),
            branch_metrics_row(case_name, actual_split, "repo_eval_select_eval_logits", data["z_repo_eval"], y),
        ])

        alpha = get_cfg_alpha(trainer)
        if np.isfinite(alpha):
            z_formula = alpha * data["z_c"] + (1.0 - alpha) * data["z_r"]
            diff_formula = np.max(np.abs(z_formula - data["z_aux_fusion"]))
        else:
            diff_formula = float("nan")

        diff_repo_fusion_eval = np.max(np.abs(data["z_repo_eval"] - data["z_aux_fusion"]))

        all_sanity_rows.append({
            "case_name": case_name,
            "requested_split_alias": split_name,
            "actual_split": actual_split,
            "n": int(len(y)),
            "alpha": alpha,
            "max_abs_formula_alpha_vs_aux_fusion": float(diff_formula),
            "formula_alpha_matches_aux_fusion_atol_1e-4": bool(np.isfinite(diff_formula) and diff_formula < 1e-4),
            "max_abs_repo_eval_vs_aux_fusion": float(diff_repo_fusion_eval),
            "repo_eval_equals_aux_fusion_atol_1e-4": bool(diff_repo_fusion_eval < 1e-4),
            **summarize_logits_matrix("z_c", data["z_c"]),
            **summarize_logits_matrix("z_r", data["z_r"]),
            **summarize_logits_matrix("z_aux_fusion", data["z_aux_fusion"]),
            **summarize_logits_matrix("z_repo_eval", data["z_repo_eval"]),
            **summarize_epistemic_data(data),
        })

        pred_c = np.argmax(data["z_c"], axis=1)
        pred_r = np.argmax(data["z_r"], axis=1)
        pred_f = np.argmax(data["z_aux_fusion"], axis=1)
        pred_eval = np.argmax(data["z_repo_eval"], axis=1)

        all_transition_rows.append({
            "case_name": case_name,
            "actual_split": actual_split,
            "n": int(len(y)),
            "C_correct_R_correct": int(np.sum((pred_c == y) & (pred_r == y))),
            "C_correct_R_wrong": int(np.sum((pred_c == y) & (pred_r != y))),
            "C_wrong_R_correct": int(np.sum((pred_c != y) & (pred_r == y))),
            "C_wrong_R_wrong": int(np.sum((pred_c != y) & (pred_r != y))),
            "C_pred_eq_R_pred": int(np.sum(pred_c == pred_r)),
            "C_pred_neq_R_pred": int(np.sum(pred_c != pred_r)),
            "fusion_changes_C_prediction": int(np.sum(pred_f != pred_c)),
            "repo_eval_changes_C_prediction": int(np.sum(pred_eval != pred_c)),
        })

        changed_idx = np.where(pred_f != pred_c)[0]
        if len(changed_idx) > 0:
            p_c = softmax_np(data["z_c"])
            p_r = softmax_np(data["z_r"])
            p_f = softmax_np(data["z_aux_fusion"])
            for i in changed_idx[:5000]:
                all_changed_rows.append({
                    "case_name": case_name,
                    "actual_split": actual_split,
                    "index": int(data["index"][i]),
                    "label": int(y[i]),
                    "pred_c": int(pred_c[i]),
                    "pred_r": int(pred_r[i]),
                    "pred_fusion": int(pred_f[i]),
                    "c_correct": bool(pred_c[i] == y[i]),
                    "r_correct": bool(pred_r[i] == y[i]),
                    "fusion_correct": bool(pred_f[i] == y[i]),
                    "conf_c": float(np.max(p_c[i])),
                    "conf_r": float(np.max(p_r[i])),
                    "conf_fusion": float(np.max(p_f[i])),
                })

        epistemic_r = get_epistemic_r_feature(data)
        epistemic_stats_split = fit_epistemic_stats(epistemic_r) if epistemic_r is not None else None
        comp = compute_dynamic_components(
            data["z_c"],
            data["z_r"],
            epistemic_r=epistemic_r,
            epistemic_stats=epistemic_stats_split,
        )
        diag_n = len(y)
        for i in range(diag_n):
            all_full_diag_rows.append({
                "case_name": case_name,
                "actual_split": actual_split,
                "index": int(data["index"][i]),
                "label": int(y[i]),
                "pred_c": int(pred_c[i]),
                "pred_r": int(pred_r[i]),
                "pred_fusion": int(pred_f[i]),
                "pred_repo_eval": int(pred_eval[i]),
                "c_correct": bool(pred_c[i] == y[i]),
                "r_correct": bool(pred_r[i] == y[i]),
                "fusion_correct": bool(pred_f[i] == y[i]),
                "repo_eval_correct": bool(pred_eval[i] == y[i]),
                "u_c": float(comp["u_c"][i]),
                "u_r": float(comp["u_r"][i]),
                "u_gap": float(comp["u_gap"][i]),
                "margin_c": float(comp["margin_c"][i]),
                "margin_r": float(comp["margin_r"][i]),
                "margin_gain": float(comp["margin_gain"][i]),
                "js_c_r": float(comp["js"][i]),
                "epistemic_r_mi": float(comp["epistemic_r_raw"][i]),
                "epistemic_r_scaled": float(comp["epistemic_r_scaled"][i]),
                "epistemic_r_rank": float(comp.get("epistemic_r_rank", np.zeros_like(y, dtype=float))[i]),
                "r_mc_pred_entropy": float(data.get("r_mc_pred_entropy", np.full_like(y, np.nan))[i]) if "r_mc_pred_entropy" in data else float("nan"),
                "r_mc_expected_entropy": float(data.get("r_mc_expected_entropy", np.full_like(y, np.nan))[i]) if "r_mc_expected_entropy" in data else float("nan"),
                "r_mc_prob_var_sum": float(data.get("r_mc_prob_var_sum", np.full_like(y, np.nan))[i]) if "r_mc_prob_var_sum" in data else float("nan"),
                "r_mc_logit_var_mean": float(data.get("r_mc_logit_var_mean", np.full_like(y, np.nan))[i]) if "r_mc_logit_var_mean" in data else float("nan"),
            })

    # beta 只用 val/beta_select split 选择，然后评估到 test。
    # prior_r = 1 - alpha，保证 beta 全 0 且 bias=0 时退化到 repo 原始 alpha 融合。
    alpha_for_prior = get_cfg_alpha(trainer)
    best_beta, beta_grid_df = grid_search_beta_on_split(
        case_name,
        val_data,
        BETA_GRID_VALUES,
        alpha=alpha_for_prior,
        use_epistemic=False,
    )
    all_beta_top_rows.append(beta_grid_df.head(50))

    best_beta_epistemic = []
    if bool(globals().get("USE_EPISTEMIC_RANK_RELIABILITY", True)) and get_epistemic_r_feature(val_data) is not None:
        for reliability_mode in list(globals().get("EPISTEMIC_RELIABILITY_MODES", ["epistemic_centered_reliability"])):
            best_beta_reliability, beta_reliability_grid_df = grid_search_beta_on_split(
                case_name,
                val_data,
                BETA_GRID_VALUES,
                alpha=alpha_for_prior,
                use_epistemic=True,
                lambda_e_values=globals().get("LAMBDA_E_GRID_VALUES", [0.0]),
                epistemic_tau_modes=globals().get("EPISTEMIC_TAU_MODES", ["median"]),
                epistemic_penalty_mode=reliability_mode,
                base_beta=best_beta,
            )
            best_beta_epistemic.append(best_beta_reliability)
            all_beta_top_rows.append(beta_reliability_grid_df.head(50))
    if not best_beta_epistemic:
        best_beta_epistemic = None

    all_accuracy_rows.extend(evaluate_dynamic_variants(
        case_name,
        test_data["actual_split"],
        test_data,
        beta=best_beta,
        beta_epistemic=best_beta_epistemic,
        alpha=alpha_for_prior,
    ))

    print("test n:", len(test_data["y"]), "actual_split:", test_data["actual_split"])
    print("C acc:", top1_acc_from_logits(test_data["z_c"], test_data["y"]))
    print("R acc:", top1_acc_from_logits(test_data["z_r"], test_data["y"]))
    print("Fusion acc:", top1_acc_from_logits(test_data["z_aux_fusion"], test_data["y"]))
    print("Repo eval acc:", top1_acc_from_logits(test_data["z_repo_eval"], test_data["y"]))
    print("best beta:", best_beta)
    if best_beta_epistemic is not None:
        print("best beta epistemic rank reliability:", best_beta_epistemic)

config_audit_df = pd.concat(all_config_audit_rows, ignore_index=True) if all_config_audit_rows else pd.DataFrame()
accuracy_df = pd.DataFrame(all_accuracy_rows)
sanity_df = pd.DataFrame(all_sanity_rows)
transitions_df = pd.DataFrame(all_transition_rows)
changed_df = pd.DataFrame(all_changed_rows)
full_diag_df = pd.DataFrame(all_full_diag_rows)
beta_top_df = pd.concat(all_beta_top_rows, ignore_index=True) if all_beta_top_rows else pd.DataFrame()

print("\nAccuracy:")
display(accuracy_df)
print("\nSanity checks:")
display(sanity_df)



CASE: MMRL_ucf101_16shot_seed1
Loading trainer: RefactorRunner
Loading dataset: UCF101
Reading split from /root/autodl-tmp/MMRL/DATASETS/ucf101/split_zhou_UCF101.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/ucf101/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  ------
Dataset    UCF101
# classes  101
# train_x  1,616
# val      404
# test     3,783
---------  ------


[MMRL] trainable params: {'representation_learner.compound_rep_tokens_r2vproj.1.bias', 'representation_learner.compound_rep_tokens_r2vproj.5.bias', 'representation_learner.compound_rep_tokens_r2vproj.6.bias', 'representation_learner.compound_rep_tokens_r2vproj.4.weight', 'representation_learner.compound_rep_tokens_r2vproj.3.weight', 'image_encoder.proj_rep', 'representation_learner.compound_rep_tokens_r2tproj.4.weight', 'representation_learner.compound_rep_tokens_r2tproj.6.bias', 'representation_learner.compound_rep_tokens_r2tproj.1.weight', 'representation_learner.compound_rep_tokens_r2vproj.0.bias', 'representation_learner.compound_rep_tokens_r2vproj.2.weight', 'representation_learner.compound_rep_tokens_r2tproj.0.bias', 'representation_learner.compound_rep_tokens_r2tproj.3.bias', 'representation_learner.compound_rep_tokens_r2tproj.4.bias', 'representation_learner.compound_rep_tokens', 'representation_learner.compound_rep_tokens_r2tproj.6.weight', 'representation_learner.compound_rep

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:72: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None
/root/miniconda3/envs/mmrl5090/lib/python3.10/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


test n: 3783 actual_split: test
C acc: 0.8638646576790907
R acc: 0.008723235527359239
Fusion acc: 0.8445678033306899
Repo eval acc: 0.8445678033306899
best beta: {'case_name': 'MMRL_ucf101_16shot_seed1', 'split': 'val', 'search_mode': 'standard', 'epistemic_penalty_mode': 'free', 'epistemic_available': False, 'beta_u': 4.0, 'beta_m': 1.0, 'beta_d': -4.0, 'beta_e': 0.0, 'lambda_e': 0.0, 'epistemic_tau_mode': 'none', 'bias': -2.0, 'prior_r': 0.30000000000000004, 'gamma': 1.0, 'tau_e': '', 'accuracy': 0.900990099009901, 'accuracy_pct': 90.0990099009901, 'n_correct': 364, 'nll': 0.3606777134674643, 'brier': 0.15243969846971744, 'ece': 0.056816460746711865, 'omega_mean': 0.0002928898601243533, 'omega_std': 0.00047727449571917564, 'omega_min': 2.4723383190483617e-05, 'omega_max': 0.003350383057651729, 'omega_raw_mean': 0.00029288986012435315, 'omega_raw_std': 0.0004772744957191748, 'epistemic_r_scaled_mean': 0.0, 'epistemic_r_scaled_std': 0.0, 'epistemic_r_rank_mean': 0.0, 'epistemic_r_rank_

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:72: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None
/root/miniconda3/envs/mmrl5090/lib/python3.10/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


test n: 3783 actual_split: test
C acc: 0.8403383558022733
R acc: 0.8419243986254296
Fusion acc: 0.8699444885011896
Repo eval acc: 0.8699444885011896
best beta: {'case_name': 'BayesMMRL_ucf101_16shot_seed1', 'split': 'val', 'search_mode': 'standard', 'epistemic_penalty_mode': 'free', 'epistemic_available': False, 'beta_u': -2.0, 'beta_m': 1.0, 'beta_d': -0.5, 'beta_e': 0.0, 'lambda_e': 0.0, 'epistemic_tau_mode': 'none', 'bias': 1.0, 'prior_r': 0.30000000000000004, 'gamma': 0.5, 'tau_e': '', 'accuracy': 0.9084158415841584, 'accuracy_pct': 90.84158415841584, 'n_correct': 367, 'nll': 0.2903922394394007, 'brier': 0.13831372414011092, 'ece': 0.02434956321812176, 'omega_mean': 0.4119571210274783, 'omega_std': 0.022066008234903872, 'omega_min': 0.32051946019729827, 'omega_max': 0.4768607530677762, 'omega_raw_mean': 0.5239142420549565, 'omega_raw_std': 0.044132016469807744, 'epistemic_r_scaled_mean': 0.0, 'epistemic_r_scaled_std': 0.0, 'epistemic_r_rank_mean': 0.0, 'epistemic_r_rank_std': 0.0, 

,case_name,split,branch,n,n_correct,accuracy,accuracy_pct,nll,brier,ece,...,epistemic_rank_center,epistemic_rank_p80,epistemic_rank_p90,epistemic_rank_p95,epistemic_reliability_mean,epistemic_reliability_std,epistemic_risk_mean,epistemic_risk_std,selected_val_temp_nll,selected_val_temp_ece
0,MMRL_ucf101_16shot_seed1,test,C_only_outputs_logits,3783,3268,0.863865,86.386466,0.468218,0.194441,0.025516,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MMRL_ucf101_16shot_seed1,test,R_only_aux_logits_rep,3783,33,0.008723,0.872324,7.613476,1.195996,0.360784,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MMRL_ucf101_16shot_seed1,test,repo_aux_logits_fusion,3783,3195,0.844568,84.456780,0.627702,0.244715,0.122207,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MMRL_ucf101_16shot_seed1,test,repo_eval_select_eval_logits,3783,3195,0.844568,84.456780,0.627702,0.244715,0.122207,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MMRL_ucf101_16shot_seed1,val,C_only_outputs_logits,404,364,0.900990,90.099010,0.360616,0.152404,0.056714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,MMRL_ucf101_16shot_seed1,val,R_only_aux_logits_rep,404,5,0.012376,1.237624,7.693477,1.191532,0.353651,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,MMRL_ucf101_16shot_seed1,val,repo_aux_logits_fusion,404,345,0.853960,85.396040,0.561858,0.228926,0.138127,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,MMRL_ucf101_16shot_seed1,val,repo_eval_select_eval_logits,404,345,0.853960,85.396040,0.561858,0.228926,0.138127,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,MMRL_ucf101_16shot_seed1,test,dynamic_no_beta_prior_centered,3783,3224,0.852234,85.223368,0.541354,0.213833,0.054756,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,MMRL_ucf101_16shot_seed1,test,dynamic_beta_shrink_selected_on_val,3783,3268,0.863865,86.386466,0.468246,0.194451,0.026079,...,,,,,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN,NaN



Sanity checks:


,case_name,requested_split_alias,actual_split,n,alpha,max_abs_formula_alpha_vs_aux_fusion,formula_alpha_matches_aux_fusion_atol_1e-4,max_abs_repo_eval_vs_aux_fusion,repo_eval_equals_aux_fusion_atol_1e-4,z_c_shape,...,z_repo_eval_mean_l2,epistemic_available,epistemic_r_mi_mean,epistemic_r_mi_std,epistemic_r_mi_p95,r_mc_num_samples_median,r_mc_pred_entropy_mean,r_mc_expected_entropy_mean,r_mc_prob_var_sum_mean,r_mc_logit_var_mean
0,MMRL_ucf101_16shot_seed1,test,test,3783,0.7,0.000000,True,0.0,True,"(3783, 101)",...,131.329681,False,,,,,NaN,NaN,NaN,NaN
1,MMRL_ucf101_16shot_seed1,beta_select,val,404,0.7,0.000000,True,0.0,True,"(404, 101)",...,129.936874,False,,,,,NaN,NaN,NaN,NaN
2,BayesMMRL_ucf101_16shot_seed1,test,test,3783,0.7,0.000008,True,0.0,True,"(3783, 101)",...,130.665955,True,0.02723,0.040571,0.118725,10.0,0.089146,0.061916,0.056206,4.693680
3,BayesMMRL_ucf101_16shot_seed1,beta_select,val,404,0.7,0.000008,True,0.0,True,"(404, 101)",...,131.475662,True,0.021752,0.036984,0.104216,10.0,0.070653,0.048901,0.044058,5.064567


## 5. 结论与 Excel 输出


In [19]:
def build_conclusions_df(accuracy_df, sanity_df, config_audit_df):
    rows = []

    rows.append({
        "item": "root_cause",
        "conclusion": "本版修复点是从真实训练日志恢复完整 args/opts；不再只从路径猜配置。",
        "detail": "如果 z_c/z_r 准确率异常低，优先检查 ConfigAudit 中 log_path、opts_restored_from_log、args.opts 与真实 run.log 是否一致。",
    })

    if not sanity_df.empty:
        for _, r in sanity_df.iterrows():
            rows.append({
                "item": "sanity",
                "conclusion": f"{r['case_name']} / {r['actual_split']} / n={r['n']}",
                "detail": (
                    f"alpha={r['alpha']}; "
                    f"max_abs_formula_alpha_vs_aux_fusion={r['max_abs_formula_alpha_vs_aux_fusion']}; "
                    f"max_abs_repo_eval_vs_aux_fusion={r['max_abs_repo_eval_vs_aux_fusion']}"
                ),
            })

    if not accuracy_df.empty:
        test_rows = accuracy_df[accuracy_df["split"].eq("test")].copy()
        for _, r in test_rows.iterrows():
            rows.append({
                "item": "accuracy",
                "conclusion": f"{r['case_name']} / {r['branch']}: {r['accuracy_pct']:.3f}%",
                "detail": f"n_correct={r['n_correct']} / n={r['n']}, nll={r['nll']:.6f}, brier={r['brier']:.6f}, ece={r['ece']:.6f}",
            })

    if not config_audit_df.empty:
        bad = config_audit_df[config_audit_df["key"].eq("opts_restored_from_log")]
        for _, r in bad.iterrows():
            rows.append({
                "item": "config_restore",
                "conclusion": f"{r['case_name']}: opts_restored_from_log={r['value']}",
                "detail": f"expected={r['expected_or_inferred']}, match={r['match']}",
            })

    return pd.DataFrame(rows)

conclusions_df = build_conclusions_df(accuracy_df, sanity_df, config_audit_df)

readme_df = pd.DataFrame([
    {
        "sheet": "README",
        "description": "说明各个 sheet 的含义。",
    },
    {
        "sheet": "Conclusions",
        "description": "自动生成的结论摘要，重点检查真实日志配置恢复和 test 分支准确率。",
    },
    {
        "sheet": "ConfigAudit",
        "description": "每个 case 从真实日志恢复的 cfg/args/opts 审计。重点看 log_path、opts_restored_from_log、args.opts。",
    },
    {
        "sheet": "Accuracy",
        "description": "C 分支、R 分支、repo fusion、repo eval、动态融合的 accuracy/NLL/Brier/ECE，以及 v4 加入的置信度/entropy/logit norm 诊断；v8 额外包含 BayesMMRL MC epistemic centered reliability 变体；full-MC sample outputs 提取 rep MC logits。",
    },
    {
        "sheet": "SanityChecks",
        "description": "shape/logit 范围、alpha 公式与 aux_fusion 的差异、repo_eval 与 aux_fusion 差异。",
    },
    {
        "sheet": "Transitions",
        "description": "C/R 分支正确性组合、fusion 是否改变 C 分支预测。",
    },
    {
        "sheet": "BetaTop",
        "description": "在 BETA_SELECT_SPLIT 上搜索出的动态融合 beta/gamma top candidates；v5/v6 先保证 correct 数接近最优，再按 NLL/ECE/omega_shift 选择；包含 standard、epistemic_rank_reliability、epistemic_margin_rank_reliability、epistemic_high_rank_veto 搜索，并拟合 temperature。",
    },
    {
        "sheet": "ChangedSamples",
        "description": "fusion 相对 C 分支改变预测的样本，最多保存前 5000 个。",
    },
    {
        "sheet": "FullDiagnostics",
        "description": "逐样本预测、正确性、u/margin/JS 诊断量；BayesMMRL 额外包含 R 分支 MC epistemic_r_mi；v9 支持 USE_MEAN_MAIN_MC_REP=False 的 full-MC 路径、predictive/expected entropy、MC 方差，以及 percentile/rank epistemic reliability。",
    },
])

report_path = OUT_DIR / "dynamic_r_fusion_repo_forward_log_restored_v9_epistemic_rank_reliability_gate_report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    readme_df.to_excel(writer, sheet_name="README", index=False)
    conclusions_df.to_excel(writer, sheet_name="Conclusions", index=False)
    config_audit_df.to_excel(writer, sheet_name="ConfigAudit", index=False)
    accuracy_df.to_excel(writer, sheet_name="Accuracy", index=False)
    sanity_df.to_excel(writer, sheet_name="SanityChecks", index=False)
    transitions_df.to_excel(writer, sheet_name="Transitions", index=False)
    beta_top_df.to_excel(writer, sheet_name="BetaTop", index=False)
    changed_df.to_excel(writer, sheet_name="ChangedSamples", index=False)
    full_diag_df.to_excel(writer, sheet_name="FullDiagnostics", index=False)

print("Saved report:", report_path)
display(conclusions_df)


Saved report: /root/autodl-tmp/MMRL/output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored_v9_epistemic_rank_reliability/dynamic_r_fusion_repo_forward_log_restored_v9_epistemic_rank_reliability_gate_report.xlsx


,item,conclusion,detail
0,root_cause,本版修复点是从真实训练日志恢复完整 args/opts；不再只从路径猜配置。,如果 z_c/z_r 准确率异常低，优先检查 ConfigAudit 中 log_path、...
1,sanity,MMRL_ucf101_16shot_seed1 / test / n=3783,alpha=0.7; max_abs_formula_alpha_vs_aux_fusion...
2,sanity,MMRL_ucf101_16shot_seed1 / val / n=404,alpha=0.7; max_abs_formula_alpha_vs_aux_fusion...
3,sanity,BayesMMRL_ucf101_16shot_seed1 / test / n=3783,alpha=0.7; max_abs_formula_alpha_vs_aux_fusion...
4,sanity,BayesMMRL_ucf101_16shot_seed1 / val / n=404,alpha=0.7; max_abs_formula_alpha_vs_aux_fusion...
5,accuracy,MMRL_ucf101_16shot_seed1 / C_only_outputs_logi...,"n_correct=3268 / n=3783, nll=0.468218, brier=0..."
6,accuracy,MMRL_ucf101_16shot_seed1 / R_only_aux_logits_r...,"n_correct=33 / n=3783, nll=7.613476, brier=1.1..."
7,accuracy,MMRL_ucf101_16shot_seed1 / repo_aux_logits_fus...,"n_correct=3195 / n=3783, nll=0.627702, brier=0..."
8,accuracy,MMRL_ucf101_16shot_seed1 / repo_eval_select_ev...,"n_correct=3195 / n=3783, nll=0.627702, brier=0..."
9,accuracy,MMRL_ucf101_16shot_seed1 / dynamic_no_beta_pri...,"n_correct=3224 / n=3783, nll=0.541354, brier=0..."


## 6. 快速人工校验

如果你期望 Caltech101 test set 为 `n=2465`，可以运行：

```python
assert sanity_df.query("actual_split == 'test'")["n"].eq(2465).all()
```

如果 R 分支不是你预期的 95% 左右，先看：

1. `ConfigAudit.log_path` 是否指向真实训练日志；
2. `ConfigAudit.args.opts` 是否包含真实训练命令里的所有覆盖项；
3. `Accuracy` 的 `split` 是否为 `test`、`n` 是否为 2465；
4. `SanityChecks.max_abs_formula_alpha_vs_aux_fusion` 是否符合该方法当前 eval aggregation 的预期。
